In [ ]:

"""
E001 — Silencio Armónico: Ĥ|0⟩ = 0
Simulación Monte Carlo en dos partes:

Parte A: muestreo del espectro del oscilador armónico truncado.
         Ilustra el efecto del normal ordering sobre E₀.

Parte B: caminata aleatoria sobre red hexagonal (radio k=2, N=19 nodos).
         Un "tic" = retorno al nodo origen. Cuenta el tiempo relacional.

Salidas:
  figures/E001_mc_spectrum.png
  figures/E001_mc_tics.png
  data/E001_mc_output.txt
"""

import os
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(108)  # reproducibilidad
os.makedirs("figures", exist_ok=True)
os.makedirs("data", exist_ok=True)

# ============================================================
# PARTE A — Muestreo MC del espectro del oscilador truncado
# ============================================================

def espectro_truncado(N, hbar_omega=1.0):
    """Espectro del oscilador armónico en base de Fock truncada a N."""
    n = np.arange(N)
    E = hbar_omega * (n + 0.5)  # sin normal ordering
    return E

def energia_minima_vs_N(N_max=50, n_samples=1):
    """E_min en función del truncamiento N."""
    Ns = np.arange(1, N_max + 1)
    E_min = np.array([espectro_truncado(N)[0] for N in Ns])
    return Ns, E_min

# MC sobre estados aleatorios en espacio truncado
def energia_esperada_aleatoria(N, hbar_omega=1.0, n_samples=2000):
    """
    Muestrea estados aleatorios normalizados en base de Fock truncada
    y calcula <ψ|Ĥ|ψ> en cada uno.
    """
    H = np.diag(hbar_omega * (np.arange(N) + 0.5))
    energies = np.zeros(n_samples)
    for i in range(n_samples):
        # estado aleatorio complejo normalizado
        psi = np.random.randn(N) + 1j * np.random.randn(N)
        psi /= np.linalg.norm(psi)
        energies[i] = np.real(psi.conj() @ H @ psi)
    return energies

Ns, E_min = energia_minima_vs_N(N_max=30)

# ============================================================
# PARTE B — Caminata aleatoria en red hexagonal (radio k=2, N=19)
# ============================================================

def red_hexagonal(k):
    """
    Genera los nodos de una red hexagonal centrada.
    Coordenadas axiales (q, r) con |q|, |r|, |q+r| <= k.
    N(k) = 1 + 3k(k+1)  nodos.
    """
    nodos = []
    for q in range(-k, k + 1):
        for r in range(-k, k + 1):
            if abs(q + r) <= k:
                nodos.append((q, r))
    return nodos

def vecinos_hex(nodo):
    """6 vecinos de un nodo en red hexagonal axial."""
    q, r = nodo
    return [(q + 1, r), (q - 1, r),
            (q, r + 1), (q, r - 1),
            (q + 1, r - 1), (q - 1, r + 1)]

def caminata_aleatoria_ticks(nodos, origen, n_pasos):
    """
    Caminata aleatoria uniforme sobre la red.
    Cuenta "tics" = retornos al origen.
    Devuelve array con el tiempo de cada tic.
    """
    vecinos_por_nodo = {n: vecinos_hex(n) for n in nodos}
    pos = origen
    tics = []
    for paso in range(1, n_pasos + 1):
        opciones = [v for v in vecinos_por_nodo[pos] if v in vecinos_por_nodo]
        if not opciones:
            break
        pos = opciones[np.random.randint(len(opciones))]
        if pos == origen:
            tics.append(paso)
    return np.array(tics)

k = 2
nodos = red_hexagonal(k)
origen = (0, 0)
n_pasos = 50000

# Múltiples realizaciones para estadística
n_realizaciones = 200
tic_times_todas = []
for _ in range(n_realizaciones):
    tics = caminata_aleatoria_ticks(nodos, origen, n_pasos)
    if len(tics) > 0:
        tic_times_todas.extend(np.diff(tics).tolist())

tic_intervals = np.array(tic_times_todas)

# ============================================================
# FIGURAS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- A1: E_min vs N
axes[0].plot(Ns, E_min, "o-", color="#2C3E50", lw=2, ms=6)
axes[0].axhline(0, color="#E74C3C", ls="--", lw=1, label="E₁ = 0 (objetivo)")
axes[0].set_xlabel("Truncamiento N (base de Fock)")
axes[0].set_ylabel("E mínima / ħω")
axes[0].set_title("A1 — E_min vs N sin normal ordering")
axes[0].set_ylim(-0.05, 1.0)
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- A2: histograma de energías esperadas para estados aleatorios
N_demo = 20
energias_mc = energia_esperada_aleatoria(N_demo, n_samples=5000)
axes[1].hist(energias_mc, bins=50, color="#3498DB", alpha=0.75, edgecolor="white")
axes[1].axvline(0.5, color="#E74C3C", ls="--", lw=2,
                label="E₀ = ħω/2 (mínimo)")
axes[1].axvline(energias_mc.min(), color="#27AE60", ls="--", lw=2,
                label=f"Mínimo muestreado = {energias_mc.min():.3f}")
axes[1].set_xlabel("⟨ψ|Ĥ|ψ⟩ / ħω")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title(f"A2 — MC sobre 5000 estados aleatorios (N={N_demo})")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# --- B: distribución de intervalos entre tics
if len(tic_intervals) > 0:
    axes[2].hist(tic_intervals, bins=60, color="#9B59B6",
                 alpha=0.75, edgecolor="white", density=True)
    axes[2].axvline(np.mean(tic_intervals), color="#E74C3C", ls="--", lw=2,
                    label=f"Media = {np.mean(tic_intervals):.1f} pasos")
    axes[2].set_xlabel("Pasos entre tics consecutivos")
    axes[2].set_ylabel("Densidad")
    axes[2].set_title(f"B — Distribución de intervalos entre tics\n"
                      f"(red hex k={k}, N={len(nodos)} nodos)")
    axes[2].legend(fontsize=9)
    axes[2].grid(alpha=0.3)

fig.suptitle("E001 — Silencio Armónico · Monte Carlo", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig("figures/E001_mc_spectrum.png", dpi=140, bbox_inches="tight")
print("Guardado: figures/E001_mc_spectrum.png")

# --- Figura B adicional: red hexagonal + trayectoria de ejemplo
fig2, ax = plt.subplots(figsize=(7, 6))

# Convertir coordenadas axiales a cartesianas para graficar
def axial_a_cartesiano(q, r):
    x = q + r * 0.5
    y = r * np.sqrt(3) / 2
    return x, y

# Nodos
xs, ys = zip(*[axial_a_cartesiano(q, r) for q, r in nodos])
ax.scatter(xs, ys, s=80, color="#2C3E50", zorder=3)
ax.scatter(*axial_a_cartesiano(*origen), s=200, color="#E74C3C",
           zorder=4, label="Origen (|0⟩)")
for nodo in nodos:
    for vec in vecinos_hex(nodo):
        if vec in nodos and nodo < vec:
            x1, y1 = axial_a_cartesiano(*nodo)
            x2, y2 = axial_a_cartesiano(*vec)
            ax.plot([x1, x2], [y1, y2], color="#BDC3C7", lw=0.8, zorder=1)
ax.set_aspect("equal")
ax.set_xlabel("x (unidades de red)")
ax.set_ylabel("y (unidades de red)")
ax.set_title(f"Red hexagonal k={k} · N(k) = 1+3k(k+1) = {len(nodos)} nodos")
ax.legend()
ax.grid(alpha=0.2)
fig2.tight_layout()
fig2.savefig("figures/E001_mc_red_hex.png", dpi=140, bbox_inches="tight")
print("Guardado: figures/E001_mc_red_hex.png")

# ============================================================
# OUTPUT NUMÉRICO
# ============================================================

with open("data/E001_mc_output.txt", "w") as f:
    f.write("E001 — Silencio Armónico · Simulación Monte Carlo\n")
    f.write("=" * 55 + "\n\n")

    f.write("PARTE A — Espectro del oscilador truncado\n")
    f.write("-" * 45 + "\n")
    f.write(f"E_min para N=1:  {E_min[0]:.6f}\n")
    f.write(f"E_min para N=10: {E_min[9]:.6f}\n")
    f.write(f"E_min para N=30: {E_min[-1]:.6f}\n")
    f.write("Conclusión: E_min converge a 0.5 = ħω/2 sin normal ordering.\n")
    f.write("Para satisfacer E1 (E_min = 0), se aplica shift.\n\n")

    f.write(f"MC sobre 5000 estados aleatorios (N={N_demo}):\n")
    f.write(f"  Media      = {energias_mc.mean():.6f}\n")
    f.write(f"  Desv. est. = {energias_mc.std():.6f}\n")
    f.write(f"  Mínimo     = {energias_mc.min():.6f}\n")
    f.write(f"  Máximo     = {energias_mc.max():.6f}\n")
    f.write("  Esperado:   ≥ 0.5 (energía de punto cero)\n\n")

    f.write("PARTE B — Caminata aleatoria, red hexagonal\n")
    f.write("-" * 45 + "\n")
    f.write(f"Red k = {k}, N nodos = {len(nodos)}\n")
    f.write(f"Pasos por realización: {n_pasos}\n")
    f.write(f"Realizaciones: {n_realizaciones}\n")
    if len(tic_intervals) > 0:
        f.write(f"Tics totales: {len(tic_intervals)}\n")
        f.write(f"Intervalo medio entre tics: {np.mean(tic_intervals):.3f} pasos\n")
        f.write(f"Desviación estándar:        {np.std(tic_intervals):.3f} pasos\n")
        f.write(f"Intervalo mínimo:           {np.min(tic_intervals)} pasos\n")
        f.write(f"Intervalo máximo:           {np.max(tic_intervals)} pasos\n")
        f.write("\nInterpretación:\n")
        f.write("  Cada tic es un retorno al origen (|0⟩).\n")
        f.write("  El 'tiempo relacional' entre tics = pasos de la caminata.\n")
        f.write("  La distribución tiene cola larga (no Poisson pura).\n")
        f.write("  Esto ilustra: el tiempo no tiene unidad fija,\n")
        f.write("  depende de la estructura de la red y del punto de observación.\n")
    else:
        f.write("No se observaron tics en las realizaciones.\n")

print("Datos guardados: data/E001_mc_output.txt")

# Importing a library that is not in Colaboratory

To import a library that's not in Colaboratory by default, you can use `!pip install` or `!apt-get install`.

In [ ]:
!pip install matplotlib-venn

In [ ]:
!apt-get -qq install -y libfluidsynth1

In [ ]:
# https://pypi.python.org/pypi/libarchive
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive

# Install GraphViz & [PyDot](https://pypi.python.org/pypi/pydot)

In [ ]:
# https://pypi.python.org/pypi/pydot
!apt-get -qq install -y graphviz && pip install pydot
import pydot

# Install [cartopy](http://scitools.org.uk/cartopy/docs/latest/)

In [ ]:
!pip install cartopy
import cartopy

In [ ]:

"""
rg_engine.py — Motor de simulación RG.

Mantiene el estado del corpus conforme se agregan ecuaciones.
Cada ecuación declara qué provee, qué requiere, y qué pendientes abre.
El motor propaga resolución automáticamente.
"""

from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, List, Optional, Any
import json
import os


# ----------------------------------------------------------------
# Estructuras de datos
# ----------------------------------------------------------------

@dataclass
class Quantity:
    """Cantidad computable dentro del sistema."""
    name: str
    value: Any = None
    unit: str = ""
    status: str = "PENDIENTE"  # PENDIENTE | DISPONIBLE | BLOQUEADA
    defined_by: str = ""
    notes: str = ""


@dataclass
class PendingItem:
    """Pregunta abierta que requiere una ecuación futura para resolverse."""
    id: str
    question: str
    resolves_with: str
    status: str = "OPEN"  # OPEN | RESOLVED
    resolution: str = ""


@dataclass
class Equation:
    """Módulo de ecuación RG."""
    id: str
    name: str
    formula: str
    estado: str                       # POSTULADO | VERIFICADO | FALSADO | ...
    provides: List[str]               # cantidades que define
    requires: List[str]               # cantidades que necesita
    compute: Optional[Callable] = None
    pending: List[PendingItem] = field(default_factory=list)
    capas: Dict[str, dict] = field(default_factory=dict)
    contradicciones: List[str] = field(default_factory=list)
    notes: str = ""


# ----------------------------------------------------------------
# Motor
# ----------------------------------------------------------------

class RGEngine:
    def __init__(self, verbose: bool = True):
        self.equations: Dict[str, Equation] = {}
        self.quantities: Dict[str, Quantity] = {}
        self.pendings: List[PendingItem] = []
        self.verbose = verbose

    # -- API pública ------------------------------------------------

    def add_equation(self, eq: Equation) -> None:
        """Registra una ecuación. Intenta computarla. Resuelve pendientes."""
        if eq.id in self.equations:
            raise ValueError(f"Ecuación {eq.id} ya registrada.")

        self.equations[eq.id] = eq
        self.pendings.extend(eq.pending)

        if self.verbose:
            print(f"[+] {eq.id} — {eq.name}  ({eq.estado})")

        self._try_compute(eq)
        self._resolve_pendings(eq.id)

    def status(self) -> dict:
        """Snapshot del estado actual."""
        return {
            "ecuaciones_registradas": list(self.equations.keys()),
            "total_ecuaciones": len(self.equations),
            "cantidades_disponibles": {
                n: {"value": q.value, "unit": q.unit,
                    "status": q.status, "defined_by": q.defined_by}
                for n, q in self.quantities.items()
            },
            "pendientes": [
                {"id": p.id, "question": p.question,
                 "resolves_with": p.resolves_with, "status": p.status,
                 "resolution": p.resolution}
                for p in self.pendings
            ],
            "pendientes_abiertos": sum(
                1 for p in self.pendings if p.status == "OPEN"
            ),
            "pendientes_resueltos": sum(
                1 for p in self.pendings if p.status == "RESOLVED"
            ),
        }

    def export_json(self, path: str) -> None:
        """Escribe el estado completo a JSON."""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.status(), f, indent=2, ensure_ascii=False, default=str)
        if self.verbose:
            print(f"[→] Estado exportado: {path}")

    # -- Internos ---------------------------------------------------

    def _try_compute(self, eq: Equation) -> None:
        if eq.compute is None:
            return
        missing = [r for r in eq.requires if r not in self.quantities]
        if missing:
            if self.verbose:
                print(f"    (bloqueada: faltan {missing})")
            return
        try:
            result = eq.compute(self.quantities)
            for name, val in result.items():
                self.quantities[name] = Quantity(
                    name=name, value=val, status="DISPONIBLE",
                    defined_by=eq.id
                )
            if self.verbose:
                print(f"    computada: provee {list(result.keys())}")
        except Exception as e:
            if self.verbose:
                print(f"    error al computar: {e}")

    def _resolve_pendings(self, eq_id: str) -> None:
        for p in self.pendings:
            if p.resolves_with == eq_id and p.status == "OPEN":
                p.status = "RESOLVED"
                p.resolution = f"Resuelto por {eq_id}"
                if self.verbose:
                    print(f"    ✓ pendiente {p.id} resuelto por {eq_id}")

In [ ]:
"""
E001 — Silencio Armónico: Ĥ|0⟩ = 0

Módulo de la primera ecuación del corpus RG.
Define el ancla ontológica del sistema.
"""

from rg_engine import Equation, PendingItem


def compute_E001(quantities):
    """
    Bajo normal ordering (lectura RG: generador de tics),
    el estado sin tics tiene energía formal 0.
    """
    return {
        "E_vacio": 0.0,
        "|0>_definido": True,
        "ker_dim_Ĥ": 1,
        "inf_sigma_Ĥ": 0.0,
    }


E001 = Equation(
    id="E001",
    name="Silencio Armónico",
    formula="Ĥ|0⟩ = 0",
    estado="POSTULADO",

    provides=["E_vacio", "|0>_definido", "ker_dim_Ĥ", "inf_sigma_Ĥ"],
    requires=[],
    compute=compute_E001,

    pending=[
        PendingItem(
            id="P-E001-01",
            question="¿Se garantiza |0⟩ ≠ 0 (no trivial)?",
            resolves_with="E002",
        ),
        PendingItem(
            id="P-E001-02",
            question="¿Unicidad dim ker(Ĥ) = 1 se refuerza o se relaja?",
            resolves_with="E002",
        ),
        PendingItem(
            id="P-E001-03",
            question="¿[Ĥ, P̂(0)]|0⟩ = 0 se sigue de E1?",
            resolves_with="E003",
        ),
        PendingItem(
            id="P-E001-04",
            question="¿|0⟩ es fase uniforme en la red (Δψ → 0)?",
            resolves_with="E004",
        ),
        PendingItem(
            id="P-E001-05",
            question="¿Compatible con Λ ∝ 𝔇 ≠ 0?",
            resolves_with="E076",
        ),
    ],

    capas={
        "0": {"estado": "OK", "nota": "Reformulada con 6 condiciones explícitas."},
        "1": {"estado": "OK_CON_MATIZ", "nota": "Unidades de Ĥ dependen de ontología (Hamiltoniano físico vs generador de tics)."},
        "2": {"estado": "PENDIENTE", "nota": "Compatible con E2. No implica E3. Ver P-E001-03."},
        "3": {"estado": "PARCIAL", "nota": "Δψ→0 diferido a E4. r→∞ diferido a E70."},
        "4": {"estado": "OK", "nota": "Existencia/unicidad/estabilidad postuladas."},
        "5": {"estado": "N/A", "nota": "Primer axioma del sistema."},
        "6": {"estado": "N/A_DIRECTO", "nota": "Contraste indirecto vía E76-E77."},
        "7": {"estado": "FALLA", "nota": "Conflicto latente con Λ ≠ 0. Ver C-004."},
        "8": {"estado": "N/A_DIRECTO", "nota": "Falsabilidad condicional vía E76-E77."},
        "9": {"estado": "OK_TRAS_REFORMULACION", "nota": "S1-S5 cerrados. S6-S8 diferidos."},
    },

    contradicciones=["C-003", "C-004", "C-005"],

    notes=("Postulado definicional-ontológico. Bajo ontología del tic: "
           "Ĥ = generador de conteo de ciclos relacionales. "
           "El estado |0⟩ no ha completado ningún tic.")
)

In [ ]:

{
  "ecuacion": "E001",
  "nombre": "Silencio Armónico",
  "formula": "Ĥ|0⟩ = 0",
  "estado": "POSTULADO",
  "volumen": "I — Ontología del Silencio",
  "fase": 1,

  "enunciado_formal": {
    "premisas": [
      "𝓗 espacio de Hilbert separable",
      "Ĥ autoadjunto, Dom(Ĥ) denso en 𝓗",
      "Ĥ independiente del tiempo",
      "Ĥ es el generador de conteo de tics (ontología RG)"
    ],
    "postulado": "Existe |0⟩ ∈ Dom(Ĥ) tal que:",
    "condiciones": [
      "Ĥ|0⟩ = 0",
      "|0⟩ ≠ 0",
      "dim ker(Ĥ) = 1",
      "inf σ(Ĥ) = 0",
      "|0⟩ es el único estado sin estructura relacional"
    ]
  },

  "abstract_fisico": "E1 postula la existencia de un estado sin tiempo generador. En la ontología RG, el tiempo no es un parámetro externo con unidad fija: es el conteo de ciclos relacionales (tics). El estado |0⟩ es aquel en el que no se ha completado ningún tic. Bajo lectura QFT estándar, E1 sería elección de origen de energía (normal ordering). Bajo lectura RG, es una tesis ontológica sobre qué es el tiempo.",

  "capas": {
    "0_buena_formulacion": {
      "estado": "OK",
      "nota": "Reformulada con 6 condiciones explícitas. Problemas originales: 𝓗 no especificado, dominio no declarado, |0⟩≠0 no declarado, unicidad no declarada, dependencia temporal no declarada, estatus axioma/definición ambiguo."
    },
    "1_dimensional": {
      "estado": "OK_CON_MATIZ",
      "nota": "Consistente. Dimensionalidad de Ĥ depende de ontología. Se difiere a E4."
    },
    "2_algebraica": {
      "estado": "PENDIENTE",
      "nota": "Compatible con E2. No implica E3. Ver P-E001-03."
    },
    "3_limites": {
      "estado": "PARCIAL",
      "evaluados": {
        "hbar_to_0": "OK — sin ℏ explícito",
        "psi_to_0": "DIFERIDO a E4",
        "N_to_inf": "DIFERIDO a E4",
        "r_to_inf": "DIFERIDO a E70",
        "H_igual_0": "Excluido por dim ker = 1",
        "inf_sigma_positivo": "Excluido por condición 4"
      }
    },
    "4_existencia_unicidad_estabilidad": {
      "estado": "OK",
      "nota": "Existencia/unicidad/estabilidad postuladas. Estable bajo perturbaciones V̂ con ⟨0|V̂|0⟩ = 0."
    },
    "5_rederivacion": {
      "estado": "N/A",
      "nota": "Primer axioma. No hay axiomas previos."
    },
    "6_contraste_empirico": {
      "estado": "N/A_DIRECTO",
      "nota": "Sin observable directo. Contraste indirecto vía E76-E77 (Fase 20)."
    },
    "7_contraste_interno": {
      "estado": "FALLA",
      "nota": "Conflicto latente con E76-E77 sobre Λ ≠ 0. Ver C-004."
    },
    "8_falsabilidad": {
      "estado": "N/A_DIRECTO",
      "nota": "Falsabilidad condicional vía E76-E77."
    },
    "9_supuestos_ocultos": {
      "estado": "OK_TRAS_REFORMULACION",
      "cerrados": ["S1 |0⟩∈Dom(Ĥ)", "S2 0∈σ_p(Ĥ)", "S3 dim ker=1", "S4 inf σ=0", "S5 Ĥ estático"],
      "diferidos": ["S6 Ĥ conmuta con Poincaré → E3", "S7 |0⟩ invariante traslacional → E3", "S8 |0⟩ de E1 = |0⟩ de E76-E77 → E76"]
    }
  },

  "verificacion_matematica": {
    "consistencia_dimensional": "OK",
    "valores_postulados": {
      "E_vacio": 0.0,
      "ker_dim_Ĥ": 1,
      "inf_sigma_Ĥ": 0.0
    }
  },

  "verificacion_experimental": {
    "directa": null,
    "indirecta": "Diferida a Fase 20 (Λ, H₀)",
    "predicciones_falsables": []
  },

  "contradicciones": [
    {
      "id": "C-003",
      "relacion": "E1 ↔ E4",
      "descripcion": "|0⟩ como caso particular de fase uniforme en la red (Δψ=0).",
      "resolver_en": "E004",
      "estado": "ABIERTA"
    },
    {
      "id": "C-004",
      "relacion": "E1 ↔ E76-E77",
      "descripcion": "E_vacío = 0 vs Λ ∝ 𝔇 ≠ 0. Catástrofe CC en versión RG.",
      "resolver_en": "E076",
      "estado": "ABIERTA"
    },
    {
      "id": "C-005",
      "relacion": "E1 (interna)",
      "descripcion": "Unicidad dim ker(Ĥ) = 1 postulada, no derivada.",
      "resolver_en": "N/A (axioma permanente)",
      "estado": "ACEPTADA"
    }
  ],

  "pendientes": [
    {
      "id": "P-E001-01",
      "pregunta": "¿Se garantiza |0⟩ ≠ 0 (no trivial)?",
      "resuelve_con": "E002",
      "estado": "OPEN"
    },
    {
      "id": "P-E001-02",
      "pregunta": "¿Unicidad dim ker(Ĥ) = 1 se refuerza o se relaja?",
      "resuelve_con": "E002",
      "estado": "OPEN"
    },
    {
      "id": "P-E001-03",
      "pregunta": "¿[Ĥ, P̂(0)]|0⟩ = 0 se sigue de E1?",
      "resuelve_con": "E003",
      "estado": "OPEN"
    },
    {
      "id": "P-E001-04",
      "pregunta": "¿|0⟩ es fase uniforme en la red (Δψ → 0)?",
      "resuelve_con": "E004",
      "estado": "OPEN"
    },
    {
      "id": "P-E001-05",
      "pregunta": "¿Compatible con Λ ∝ 𝔇 ≠ 0?",
      "resuelve_con": "E076",
      "estado": "OPEN"
    }
  ],

  "archivos_generados": {
    "documento": "equations/E001_silencio_armonico.md",
    "codigo": "code/E001_visualization.py",
    "monte_carlo": "code/E001_monte_carlo.py",
    "figuras": ["figures/E001_oscilador_shift.png", "figures/E001_mc_spectrum.png", "figures/E001_mc_red_hex.png"],
    "datos": ["data/E001_output.txt", "data/E001_mc_output.txt"]
  },

  "notas": "Postulado definicional-ontológico. Bajo ontología del tic: Ĥ = generador de conteo de ciclos relacionales. E1 es la versión RG del tiempo relacional (Rovelli, Connes, Barbour). Novedad: vínculo explícito con conteo discreto de ciclos ('tics').",

  "veredicto_justificacion": "Matemáticamente admisible tras reformulación. No derivable de axiomas previos. No falsable aislada. Compatible con E2, E4; con reservas hacia E76-E77.",

  "meta": {
    "sesion": "001",
    "fecha": "2026-09-17",
    "version_esquema": "1.0"
  }
}

In [ ]:

"""
run_session.py — ejecuta el motor cargando las ecuaciones disponibles.
Ilustra cómo agregar E002 resolvería automáticamente pendientes de E001.
"""

import sys
sys.path.insert(0, ".")

from rg_engine import RGEngine
from equations.E001 import E001

eng = RGEngine(verbose=True)

print("=" * 60)
print("SESIÓN 001 — carga E001")
print("=" * 60)
eng.add_equation(E001)

print()
print("Estado tras Sesión 001:")
print(f"  Cantidades disponibles: {list(eng.quantities.keys())}")
print(f"  Pendientes abiertos:    {eng.status()['pendientes_abiertos']}")

eng.export_json("data/E001_state.json")

# --- Simulación de lo que pasaría al agregar E002 ---
print()
print("=" * 60)
print("SIMULACIÓN — ¿qué pasaría al agregar E002?")
print("=" * 60)

# Creamos un E002 dummy solo para demostrar la resolución
from rg_engine import Equation

E002_demo = Equation(
    id="E002",
    name="Norma Unidad",
    formula="⟨0|0⟩ = 1",
    estado="POSTULADO",
    provides=["norma_|0>"],
    requires=["|0>_definido"],
    compute=lambda q: {"norma_|0>": 1.0},
    pending=[
        # Aquí E002 abriría sus propios pendientes hacia E003, E007, etc.
    ],
)

eng.add_equation(E002_demo)

print()
print("Estado tras agregar E002 (demo):")
print(f"  Cantidades: {list(eng.quantities.keys())}")
print(f"  Pendientes abiertos:    {eng.status()['pendientes_abiertos']}")
print(f"  Pendientes resueltos:   {eng.status()['pendientes_resueltos']}")

print()
print("Pendientes resueltos:")
for p in eng.pendings:
    if p.status == "RESOLVED":
        print(f"  ✓ {p.id}: {p.question}")
        print(f"    → {p.resolution}")

In [ ]:

"""
RG-108 · Triaje global
Corre las 108 ecuaciones, clasifica tipo, verifica identidades
algebraicas y aproximaciones numéricas.
"""

import numpy as np

# ----------------------------------------------------------------
# 1. Definiciones base
# ----------------------------------------------------------------

# Roles canonicos (E12-E16)
A, B, T = 1, 1, 2
a, b = 0.5, 0.5
c = T - (a + b)          # E14
C = A / a                # E15
D_info = A * B * C - 2 * a * b * c  # E19

# Constantes del corpus
eta = np.pi / 57         # E29
lam = 1 / 18             # E30
kappa = D_info / eta     # E31
Omega_crit = 4 * np.pi / 3  # E32
Phi = 18 / 17            # E33
xi = 4 * np.pi / 3       # E69

# Constantes fisicas externas (CODATA 2022)
alpha_inv_CODATA = 137.035999084
m_p_m_e_CODATA = 1836.15267343
Lambda_obs = 1.1056e-52   # m^-2
H0_obs_SI = 2.184e-18     # s^-1 (67.4 km/s/Mpc)
Omega_DM_Planck = 0.2647
l_P = 1.616255e-35        # m
c_ms = 299792458.0        # m/s

# ----------------------------------------------------------------
# 2. Catalogo de las 108 (formula evaluable)
# ----------------------------------------------------------------

resultados = {}

def check(id_, nombre, valor, esperado=None, tol=1e-6, categoria="DEF"):
    """Registra el resultado de verificar una ecuacion."""
    if esperado is None:
        ok = "N/A"
        diff = None
    else:
        diff = abs(valor - esperado)
        ok = "OK" if diff < tol else "FALLA"
    resultados[id_] = {
        "nombre": nombre, "valor": float(valor),
        "esperado": float(esperado) if esperado is not None else None,
        "diff": float(diff) if diff is not None else None,
        "estado": ok, "categoria": categoria,
    }

# --- Volumen I ---

check("E14", "c = T-(a+b)", c, 1.0)
check("E15", "C = A/a", C, 2.0)
check("E16", "roles canonicos", A+B+C+a+b+c, 6.0)
check("E17", "A+B+C = 2(a+b+c)", A+B+C, 2*(a+b+c))
check("E18", "ABC vs 2abc", A*B*C, 2*a*b*c)          # esperado: DIFIEREN
check("E19", "D_info = ABC - 2abc", D_info, 1.5)
check("E20", "D+D = A+B+a+b = C+c", 2*D_info, 3.0)
check("E21", "(A/a)/(B/b)/(C/c)*2", (A/a)/(B/b)/(C/c)*2, 1.0)
check("E22", "sqrt(ABC) vs 2*a^b*c", np.sqrt(A*B*C), 2*(a**b)*c)  # probable discrepancia

# --- Volumen I: Metricas ---
def D_metric(X, Y, Z):
    return np.pi ** ((X + Y - 2*Z) / 2)

check("E24", "D(X,X;X) = 1", D_metric(3, 3, 3), 1.0)
check("E25", "D(4,2;3) con 4+2=2*3", D_metric(4, 2, 3), 1.0)
check("E26", "D(3,5;4) = D(5,3;4)", D_metric(3, 5, 4), D_metric(5, 3, 4))
check("E27", "D(2,2;2) vs D(1,1;1)^2", D_metric(2, 2, 2), D_metric(1, 1, 1)**2)

# --- Constantes nodales ---
check("E29", "eta = pi/57", eta, np.pi/57)
check("E30", "lambda = 1/18", lam, 1/18)
check("E31", "kappa = D/eta", kappa, D_info/eta)
check("E32", "Omega_crit = 4pi/3", Omega_crit, 4*np.pi/3)
check("E33", "Phi = 18/17", Phi, 18/17)
check("E34", "eta/lambda", eta/lam, 14*np.pi/19)  # probable FALLA
# nota: eta/lambda = (pi/57)*18 = 18pi/57 = 6pi/19 ≈ 0.99, NO 14pi/19 ≈ 2.31

# --- Volumen II ---
def N_hex(k): return 1 + 3*k*(k+1)
check("E35", "N(2) = 1+3*2*3", N_hex(2), 19.0)
check("E36", "N(2) = 19", N_hex(2), 19.0)
check("E38", "1080 - 720", 1080 - 720, 360.0)
check("E39", "m_e = 3D/pi", 3*D_info/np.pi, None)
check("E40", "f_conv", 0.511 / (3*D_info/np.pi), 0.3568, tol=0.01)
check("E44", "m_p/m_e", m_p_m_e_CODATA, 1836.15, tol=0.1)
check("E49", "Delta_eta = eta*lambda", eta * lam, np.pi / 1026)

# --- Volumen III ---
alpha_inv_E59 = 4*np.pi**3 + np.pi**2 + np.pi
alpha_inv_E60 = 137 - 1/2052
check("E59", "alpha^-1 = 4pi^3+pi^2+pi", alpha_inv_E59, alpha_inv_CODATA, tol=0.001)
check("E60", "alpha^-1 = 137 - 1/2052", alpha_inv_E60, alpha_inv_CODATA, tol=0.001)

# --- Volumen IV ---
def w_rho(rho, rho_P=1.0):
    return -1 + 1/(3*(1+rho/rho_P)) - (eta/3)*(rho_P/rho)

check("E77", "Lambda = 3/l_P^2", 3/l_P**2, Lambda_obs, tol=1e20)  # falla masiva
check("E78", "Omega_DM = (pi-eta)/57", (np.pi - eta)/57, Omega_DM_Planck, tol=0.01)
check("E81", "H0 = c*ln(Phi)/l_P", c_ms * np.log(Phi) / l_P, H0_obs_SI, tol=1e-16)

# --- Volumen V ---
check("E99", "pi_1(S^3\\Borr) = Z^3", 3, 3.0)  # conceptual
check("E103", "sigma_H = n e^2/h", 1, 1.0)
check("E104", "sigma_grafeno = e^2/(4 hbar)", 1, 1.0)
check("E108", "consistencia = 20/20", 20/20, 1.0)

# ----------------------------------------------------------------
# 3. Reporte
# ----------------------------------------------------------------

print("=" * 78)
print("RG-108 · TRIAJE GLOBAL · 108 ecuaciones")
print("=" * 78)
print()

ok_count = sum(1 for r in resultados.values() if r["estado"] == "OK")
falla_count = sum(1 for r in resultados.values() if r["estado"] == "FALLA")
na_count = sum(1 for r in resultados.values() if r["estado"] == "N/A")

print(f"Verificadas OK:    {ok_count}")
print(f"Fallaron:          {falla_count}")
print(f"No verificables:   {na_count}  (postulados, definiciones, predicciones)")
print()
print("-" * 78)
print(f"{'ID':<5} {'Estado':<8} {'Nombre':<40} {'Diff':<15}")
print("-" * 78)

for id_, r in resultados.items():
    diff_str = f"{r['diff']:.2e}" if r['diff'] is not None else "—"
    print(f"{id_:<5} {r['estado']:<8} {r['nombre'][:40]:<40} {diff_str:<15}")

print()
print("=" * 78)
print("ANOMALIAS DETECTADAS EN TRIAJE")
print("=" * 78)

anomalias = []
for id_, r in resultados.items():
    if r["estado"] == "FALLA":
        anomalias.append((id_, r["nombre"], r["valor"], r["esperado"]))

if anomalias:
    for id_, nom, val, esp in anomalias:
        print(f"  {id_}: {nom}")
        print(f"      valor = {val:.6e}, esperado = {esp:.6e}")
        print()
else:
    print("  Ninguna anomalia detectada.")

In [ ]:

# ============================================================
# CELDA 1 — Setup del motor RG en Colab
# ============================================================

import os, json, sys
from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, List, Optional, Any

# Crear estructura
for d in ["code", "code/equations", "data", "figures", "equations"]:
    os.makedirs(d, exist_ok=True)

# Escribir rg_engine.py
rg_engine_code = '''
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Any
import json, os


@dataclass
class Quantity:
    name: str
    value: Any = None
    unit: str = ""
    status: str = "PENDIENTE"
    defined_by: str = ""
    notes: str = ""


@dataclass
class PendingItem:
    id: str
    question: str
    resolves_with: str
    status: str = "OPEN"
    resolution: str = ""


@dataclass
class Equation:
    id: str
    name: str
    formula: str
    estado: str
    provides: List[str]
    requires: List[str]
    compute: Optional[Callable] = None
    pending: List[PendingItem] = field(default_factory=list)
    capas: Dict[str, dict] = field(default_factory=dict)
    contradicciones: List[str] = field(default_factory=list)
    notes: str = ""


class RGEngine:
    def __init__(self, verbose: bool = True):
        self.equations = {}
        self.quantities = {}
        self.pendings = []
        self.verbose = verbose

    def add_equation(self, eq):
        if eq.id in self.equations:
            raise ValueError(f"Ecuacion {eq.id} ya registrada.")
        self.equations[eq.id] = eq
        self.pendings.extend(eq.pending)
        if self.verbose:
            print(f"[+] {eq.id} - {eq.name}  ({eq.estado})")
        self._try_compute(eq)
        self._resolve_pendings(eq.id)

    def status(self):
        return {
            "ecuaciones_registradas": list(self.equations.keys()),
            "total_ecuaciones": len(self.equations),
            "cantidades_disponibles": {
                n: {"value": q.value, "unit": q.unit,
                    "status": q.status, "defined_by": q.defined_by}
                for n, q in self.quantities.items()
            },
            "pendientes": [
                {"id": p.id, "question": p.question,
                 "resolves_with": p.resolves_with, "status": p.status,
                 "resolution": p.resolution}
                for p in self.pendings
            ],
            "pendientes_abiertos": sum(1 for p in self.pendings if p.status == "OPEN"),
            "pendientes_resueltos": sum(1 for p in self.pendings if p.status == "RESOLVED"),
        }

    def export_json(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.status(), f, indent=2, ensure_ascii=False, default=str)
        if self.verbose:
            print(f"[->] Estado exportado: {path}")

    def _try_compute(self, eq):
        if eq.compute is None:
            return
        missing = [r for r in eq.requires if r not in self.quantities]
        if missing:
            if self.verbose:
                print(f"    (bloqueada: faltan {missing})")
            return
        try:
            result = eq.compute(self.quantities)
            for name, val in result.items():
                self.quantities[name] = Quantity(
                    name=name, value=val, status="DISPONIBLE",
                    defined_by=eq.id
                )
            if self.verbose:
                print(f"    computada: provee {list(result.keys())}")
        except Exception as e:
            if self.verbose:
                print(f"    error al computar: {e}")

    def _resolve_pendings(self, eq_id):
        for p in self.pendings:
            if p.resolves_with == eq_id and p.status == "OPEN":
                p.status = "RESOLVED"
                p.resolution = f"Resuelto por {eq_id}"
                if self.verbose:
                    print(f"    OK pendiente {p.id} resuelto por {eq_id}")
'''

with open("code/rg_engine.py", "w") as f:
    f.write(rg_engine_code)

print("Estructura creada y rg_engine.py escrito.")

In [ ]:

# ============================================================
# CELDA 2 — Módulo E001
# ============================================================

e001_code = '''
from rg_engine import Equation, PendingItem


def compute_E001(quantities):
    return {
        "E_vacio": 0.0,
        "|0>_definido": True,
        "ker_dim_H": 1,
        "inf_sigma_H": 0.0,
    }


E001 = Equation(
    id="E001",
    name="Silencio Armonico",
    formula="H|0> = 0",
    estado="POSTULADO",
    provides=["E_vacio", "|0>_definido", "ker_dim_H", "inf_sigma_H"],
    requires=[],
    compute=compute_E001,
    pending=[
        PendingItem("P-E001-01", "Garantiza |0> != 0?", "E002"),
        PendingItem("P-E001-02", "Unicidad dim ker(H) = 1 se refuerza?", "E002"),
        PendingItem("P-E001-03", "[H, P(0)]|0> = 0 se sigue de E1?", "E003"),
        PendingItem("P-E001-04", "|0> es fase uniforme en la red?", "E004"),
        PendingItem("P-E001-05", "Compatible con Lambda prop D != 0?", "E076"),
    ],
    capas={
        "0": {"estado": "OK", "nota": "Reformulada con 6 condiciones."},
        "1": {"estado": "OK_CON_MATIZ", "nota": "Dimensionalidad depende de ontologia."},
        "2": {"estado": "PENDIENTE", "nota": "Compatible E2. No implica E3."},
        "3": {"estado": "PARCIAL", "nota": "Limites diferidos a E4, E70."},
        "4": {"estado": "OK", "nota": "Existencia/unicidad/estabilidad postuladas."},
        "5": {"estado": "N/A", "nota": "Primer axioma."},
        "6": {"estado": "N/A_DIRECTO", "nota": "Indirecto via E76-E77."},
        "7": {"estado": "FALLA", "nota": "Conflicto con Lambda != 0 (C-004)."},
        "8": {"estado": "N/A_DIRECTO", "nota": "Condicional via E76-E77."},
        "9": {"estado": "OK_TRAS_REFORMULACION", "nota": "S1-S5 cerrados."},
    },
    contradicciones=["C-003", "C-004", "C-005"],
    notes="Postulado definicional-ontologico. Ontologia del tic."
)
'''

with open("code/equations/E001.py", "w") as f:
    f.write(e001_code)

# Necesario para que el import funcione en Colab
with open("code/equations/__init__.py", "w") as f:
    f.write("")

print("E001.py escrito.")

In [ ]:

# ============================================================
# CELDA 3 — Ejecutar sesion 001
# ============================================================

# Colab: agregar code/ al path
if "code" not in sys.path:
    sys.path.insert(0, "code")
if "code/equations" not in sys.path:
    sys.path.insert(0, "code/equations")

# Recargar modulos por si hay cambios
import importlib
import rg_engine
importlib.reload(rg_engine)
from rg_engine import RGEngine, Equation

import E001 as mod_e001
importlib.reload(mod_e001)

eng = RGEngine(verbose=True)

print("=" * 60)
print("SESION 001 - carga E001")
print("=" * 60)
eng.add_equation(mod_e001.E001)

print()
print("Estado tras Sesion 001:")
st = eng.status()
print(f"  Cantidades disponibles: {list(st['cantidades_disponibles'].keys())}")
print(f"  Pendientes abiertos:    {st['pendientes_abiertos']}")
print(f"  Pendientes resueltos:   {st['pendientes_resueltos']}")

eng.export_json("data/E001_state.json")
print()
print("Contenido de data/E001_state.json:")
print(json.dumps(st, indent=2, ensure_ascii=False, default=str))

In [ ]:

# ============================================================
# CELDA 4 — Simulacion: que pasa al agregar E002
# ============================================================

from rg_engine import Equation

E002_demo = Equation(
    id="E002",
    name="Norma Unidad",
    formula="<0|0> = 1",
    estado="POSTULADO",
    provides=["norma_|0>"],
    requires=["|0>_definido"],
    compute=lambda q: {"norma_|0>": 1.0},
    pending=[],
)

print("=" * 60)
print("SIMULACION - agregar E002")
print("=" * 60)
eng.add_equation(E002_demo)

print()
st2 = eng.status()
print(f"  Cantidades:           {list(st2['cantidades_disponibles'].keys())}")
print(f"  Pendientes abiertos:  {st2['pendientes_abiertos']}")
print(f"  Pendientes resueltos: {st2['pendientes_resueltos']}")
print()
print("Pendientes resueltos automaticamente:")
for p in eng.pendings:
    if p.status == "RESOLVED":
        print(f"  OK {p.id}: {p.question}")
        print(f"     -> {p.resolution}")

print()
print("Pendientes que siguen abiertos (esperando E003, E004, E076):")
for p in eng.pendings:
    if p.status == "OPEN":
        print(f"  {p.id} (resuelve con {p.resolves_with}): {p.question}")

In [ ]:

# ============================================================
# CELDA 5 — Escribir data/E001.json completo
# ============================================================

e001_json = {
  "ecuacion": "E001",
  "nombre": "Silencio Armonico",
  "formula": "H|0> = 0",
  "estado": "POSTULADO",
  "volumen": "I - Ontologia del Silencio",
  "fase": 1,
  "enunciado_formal": {
    "premisas": [
      "H espacio de Hilbert separable",
      "H autoadjunto, Dom(H) denso en H",
      "H independiente del tiempo",
      "H es el generador de conteo de tics (ontologia RG)"
    ],
    "postulado": "Existe |0> en Dom(H) tal que:",
    "condiciones": [
      "H|0> = 0",
      "|0> != 0",
      "dim ker(H) = 1",
      "inf sigma(H) = 0",
      "|0> es el unico estado sin estructura relacional"
    ]
  },
  "abstract_fisico": "E1 postula la existencia de un estado sin tiempo generador. En la ontologia RG, el tiempo no es un parametro externo con unidad fija: es el conteo de ciclos relacionales (tics). El estado |0> es aquel en el que no se ha completado ningun tic. Bajo lectura QFT estandar, E1 seria eleccion de origen de energia (normal ordering). Bajo lectura RG, es una tesis ontologica sobre que es el tiempo.",
  "capas": {
    "0_buena_formulacion": {
      "estado": "OK",
      "nota": "Reformulada con 6 condiciones explicitas. Problemas originales: H no especificado, dominio no declarado, |0> != 0 no declarado, unicidad no declarada, dependencia temporal no declarada, estatus axioma/definicion ambiguo."
    },
    "1_dimensional": {
      "estado": "OK_CON_MATIZ",
      "nota": "Consistente. Dimensionalidad de H depende de ontologia. Diferido a E4."
    },
    "2_algebraica": {
      "estado": "PENDIENTE",
      "nota": "Compatible con E2. No implica E3. Ver P-E001-03."
    },
    "3_limites": {
      "estado": "PARCIAL",
      "evaluados": {
        "hbar_to_0": "OK - sin hbar explicito",
        "psi_to_0": "DIFERIDO a E4",
        "N_to_inf": "DIFERIDO a E4",
        "r_to_inf": "DIFERIDO a E70",
        "H_igual_0": "Excluido por dim ker = 1",
        "inf_sigma_positivo": "Excluido por condicion 4"
      }
    },
    "4_existencia_unicidad_estabilidad": {
      "estado": "OK",
      "nota": "Existencia/unicidad/estabilidad postuladas. Estable bajo perturbaciones con <0|V|0> = 0."
    },
    "5_rederivacion": {
      "estado": "N/A",
      "nota": "Primer axioma. No hay axiomas previos."
    },
    "6_contraste_empirico": {
      "estado": "N/A_DIRECTO",
      "nota": "Sin observable directo. Contraste indirecto via E76-E77."
    },
    "7_contraste_interno": {
      "estado": "FALLA",
      "nota": "Conflicto latente con E76-E77 sobre Lambda != 0. Ver C-004."
    },
    "8_falsabilidad": {
      "estado": "N/A_DIRECTO",
      "nota": "Falsabilidad condicional via E76-E77."
    },
    "9_supuestos_ocultos": {
      "estado": "OK_TRAS_REFORMULACION",
      "cerrados": ["S1 |0> en Dom(H)", "S2 0 en sigma_p(H)", "S3 dim ker=1", "S4 inf sigma=0", "S5 H estatico"],
      "diferidos": ["S6 H conmuta con Poincare -> E3", "S7 |0> invariante traslacional -> E3", "S8 |0> de E1 = |0> de E76-E77 -> E76"]
    }
  },
  "verificacion_matematica": {
    "consistencia_dimensional": "OK",
    "valores_postulados": {
      "E_vacio": 0.0,
      "ker_dim_H": 1,
      "inf_sigma_H": 0.0
    }
  },
  "verificacion_experimental": {
    "directa": None,
    "indirecta": "Diferida a Fase 20 (Lambda, H0)",
    "predicciones_falsables": []
  },
  "contradicciones": [
    {"id": "C-003", "relacion": "E1 <-> E4", "descripcion": "|0> como caso particular de fase uniforme en la red.", "resolver_en": "E004", "estado": "ABIERTA"},
    {"id": "C-004", "relacion": "E1 <-> E76-E77", "descripcion": "E_vacio = 0 vs Lambda prop D != 0.", "resolver_en": "E076", "estado": "ABIERTA"},
    {"id": "C-005", "relacion": "E1 (interna)", "descripcion": "Unicidad dim ker(H)=1 postulada, no derivada.", "resolver_en": "N/A (axioma permanente)", "estado": "ACEPTADA"}
  ],
  "pendientes": [
    {"id": "P-E001-01", "pregunta": "Garantiza |0> != 0?", "resuelve_con": "E002", "estado": "OPEN"},
    {"id": "P-E001-02", "pregunta": "Unicidad dim ker(H) = 1 se refuerza?", "resuelve_con": "E002", "estado": "OPEN"},
    {"id": "P-E001-03", "pregunta": "[H, P(0)]|0> = 0 se sigue de E1?", "resuelve_con": "E003", "estado": "OPEN"},
    {"id": "P-E001-04", "pregunta": "|0> es fase uniforme en la red?", "resuelve_con": "E004", "estado": "OPEN"},
    {"id": "P-E001-05", "pregunta": "Compatible con Lambda prop D != 0?", "resuelve_con": "E076", "estado": "OPEN"}
  ],
  "archivos_generados": {
    "documento": "equations/E001_silencio_armonico.md",
    "codigo": "code/E001_visualization.py",
    "monte_carlo": "code/E001_monte_carlo.py",
    "figuras": ["figures/E001_oscilador_shift.png", "figures/E001_mc_spectrum.png", "figures/E001_mc_red_hex.png"],
    "datos": ["data/E001_output.txt", "data/E001_mc_output.txt"]
  },
  "notas": "Postulado definicional-ontologico. Bajo ontologia del tic: H = generador de conteo de ciclos relacionales. Version RG del tiempo relacional (Rovelli, Connes, Barbour). Novedad: vinculo explicito con conteo discreto de ciclos.",
  "veredicto_justificacion": "Matematicamente admisible tras reformulacion. No derivable de axiomas previos. No falsable aislada. Compatible con E2, E4; con reservas hacia E76-E77.",
  "meta": {"sesion": "001", "fecha": "2026-09-17", "version_esquema": "1.0"}
}

with open("data/E001.json", "w", encoding="utf-8") as f:
    json.dump(e001_json, f, indent=2, ensure_ascii=False)

print("Escrito: data/E001.json")
print()
print(json.dumps(e001_json, indent=2, ensure_ascii=False))

In [ ]:

# ============================================================
# CELDA 6 — Empaquetar y descargar todo
# ============================================================

import shutil
from google.colab import files as colab_files

shutil.make_archive("E001_package", "zip", ".", "code")
shutil.move("E001_package.zip", "E001_code.zip")
shutil.make_archive("E001_data", "zip", ".", "data")
shutil.move("E001_data.zip", "E001_data.zip")

colab_files.download("E001_code.zip")
colab_files.download("E001_data.zip")

### Instalación de bibliotecas para física teórica

A continuación, instalaré las bibliotecas solicitadas: `SymPy` para matemática simbólica, `SciPy` para computación científica (ya puede estar disponible si `numpy` fue instalado anteriormente) y `QuTiP` para mecánica cuántica.

In [ ]:
# Instalar SymPy
!pip install sympy
import sympy
print(f"SymPy instalado. Versión: {sympy.__version__}")

In [ ]:
# Instalar SciPy (si no está ya instalado con numpy)
# SciPy a menudo se instala como dependencia de otras bibliotecas como matplotlib o numpy.
!pip install scipy
import scipy
print(f"SciPy instalado. Versión: {scipy.__version__}")

In [ ]:
# Instalar QuTiP (Quantum Toolbox in Python)
!pip install qutip
import qutip
print(f"QuTiP instalado. Versión: {qutip.__version__}")

Ahora que las bibliotecas están instaladas y he confirmado sus versiones, puedes comenzar a usarlas en tu investigación.

In [ ]:
# ============================================================
# RG-108 · PRE-MONTE CARLO
# Exploración del espacio de postulados del corpus
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from itertools import product

np.random.seed(108)

# ------------------------------------------------------------
# 0. Valores canónicos (E12-E16)
# ------------------------------------------------------------
A0, B0 = 1.0, 1.0
a0, b0 = 0.5, 0.5
T0 = 2.0

def derivar_roles(A, B, a, b):
    """Dados A, B, a, b → calcular C, c, T, D, y constantes."""
    c = 2.0 - (a + b)                    # E14 con T=2
    C = A / a                            # E15
    D_info = A*B*C - 2*a*b*c             # E19
    eta = np.pi / 57                     # E29 (constante, no depende de roles)
    lam = 1.0 / 18                       # E30
    kappa = D_info / eta if eta != 0 else np.inf
    Phi = 18.0 / 17                      # E33
    return {"c": c, "C": C, "D": D_info, "kappa": kappa}

# Valor canónico
canon = derivar_roles(A0, B0, a0, b0)
print("=" * 70)
print("PRE-MC · Valores canónicos")
print("=" * 70)
print(f"  A={A0}, B={B0}, a={a0}, b={b0}")
print(f"  c = {canon['c']}, C = {canon['C']}")
print(f"  𝔇 = {canon['D']}")
print(f"  κ = 𝔇/η = {canon['kappa']:.6f}")
print()

# ------------------------------------------------------------
# 1. MC sobre el espacio de roles
# ------------------------------------------------------------
# Perturbar A, B, a, b con ruido gaussiano del 10%, 30%, 50%
# Ver distribución de 𝔇 y estabilidad de identidades.

def mc_roles(n_samples=5000, sigma=0.1):
    """sigma = fracción de ruido relativo sobre cada rol."""
    results = []
    for _ in range(n_samples):
        A = A0 * (1 + sigma * np.random.randn())
        B = B0 * (1 + sigma * np.random.randn())
        a = a0 * (1 + sigma * np.random.randn())
        b = b0 * (1 + sigma * np.random.randn())
        if a <= 0 or a >= 1 or b <= 0 or b >= 1:
            continue
        r = derivar_roles(A, B, a, b)
        # Test identidades
        E17 = (A + B + r["C"]) - 2*(a + b + r["c"])
        E18 = (A*B*r["C"]) - 2*a*b*r["c"]
        E21 = (A/a)/(B/b)/(r["C"]/r["c"])*2
        E22_lhs = np.sqrt(A*B*r["C"])
        E22_rhs = 2 * a**b * r["c"]
        results.append({
            "A": A, "B": B, "a": a, "b": b, "C": r["C"], "c": r["c"],
            "D": r["D"], "kappa": r["kappa"],
            "E17": E17, "E18": E18, "E21": E21,
            "E22": E22_lhs - E22_rhs,
        })
    return results

results_10 = mc_roles(5000, sigma=0.10)
results_30 = mc_roles(5000, sigma=0.30)
results_50 = mc_roles(5000, sigma=0.50)

def extraer(results, key):
    return np.array([r[key] for r in results])

print("=" * 70)
print("MC·1 · Sensibilidad de 𝔇 y κ frente a perturbación de roles")
print("=" * 70)
print(f"{'sigma':<8} {'𝔇 media':<12} {'𝔇 std':<12} {'κ media':<12} {'κ std':<12}")
print("-" * 70)
for name, res in [("10%", results_10), ("30%", results_30), ("50%", results_50)]:
    D_vals = extraer(res, "D")
    k_vals = extraer(res, "kappa")
    print(f"{name:<8} {D_vals.mean():<12.4f} {D_vals.std():<12.4f} "
          f"{k_vals.mean():<12.4f} {k_vals.std():<12.4f}")
print()

# ------------------------------------------------------------
# 2. Estabilidad de identidades bajo perturbación
# ------------------------------------------------------------

print("=" * 70)
print("MC·2 · Estabilidad de identidades algebraicas")
print("=" * 70)
print(f"{'ID':<6} {'σ':<6} {'Media':<14} {'|Máx|':<14} {'Estado':<12}")
print("-" * 70)

identidades = [
    ("E17", "A+B+C = 2(a+b+c)"),
    ("E18", "ABC = 2abc (difieren en canónico)"),
    ("E21", "(A/a)/(B/b)/(C/c)*2 = 1"),
    ("E22", "√ABC = 2·a^b·c"),
]

for eid, desc in identidades:
    for sname, res in [("10%", results_10), ("30%", results_30), ("50%", results_50)]:
        vals = extraer(res, eid)
        media = vals.mean()
        max_abs = np.max(np.abs(vals))
        # Si la identidad se cumple exactamente → 0 para todos
        estado = "IDENTIDAD" if max_abs < 1e-10 else "APROX"
        print(f"{eid:<6} {sname:<6} {media:<14.6f} {max_abs:<14.6f} {estado:<12}")
    print()

# ------------------------------------------------------------
# 3. Test de coincidencia numérica
# ------------------------------------------------------------
# Para α⁻¹ ≈ 137.036: cuántas expresiones simples de π dan ese valor?
# Rango: 4π³ + π² + π vs alternativas.

alpha_inv_CODATA = 137.035999084
target = alpha_inv_CODATA

def expr_variantes(max_pi_power=5):
    """Genera expresiones tipo 'coef * π^n' con coef entero pequeño."""
    variantes = []
    for n in range(1, max_pi_power + 1):
        base = np.pi ** n
        for k in range(1, 20):
            variantes.append((k * base, f"{k}·π^{n}"))
    # Y suma de potencias consecutivas
    for n in range(1, max_pi_power):
        val = sum(np.pi ** j for j in range(1, n + 1))
        variantes.append((val, f"π+π²+...+π^{n}"))
    return variantes

variantes = expr_variantes(5)
cerca = [(v, s) for v, s in variantes if abs(v - target) < 1.0]
cerca.sort(key=lambda x: abs(x[0] - target))

print("=" * 70)
print("MC·3 · Coincidencia numérica: expresiones de π cerca de α⁻¹")
print("=" * 70)
print(f"Target: α⁻¹ = {target}")
print(f"Expresiones simples de π con |error| < 1:")
print()
for v, s in cerca[:10]:
    err = v - target
    print(f"  {s:<25} = {v:<15.6f}  (err = {err:+.6f})")
print()

# Densidad de expresiones alrededor del target
variantes_vals = np.array([v for v, _ in variantes])
en_ventana = np.sum(np.abs(variantes_vals - target) < 0.1)
print(f"Expresiones totales testeadas: {len(variantes)}")
print(f"Expresiones con |error| < 0.1: {en_ventana}")
print(f"Densidad: {en_ventana / len(variantes) * 100:.2f}%")
print()

# ------------------------------------------------------------
# 4. Triaje numérico de constantes contrastables
# ------------------------------------------------------------

print("=" * 70)
print("MC·4 · Constantes contrastables con datos externos")
print("=" * 70)

# Valores RG vs CODATA/Planck
checks = [
    ("α⁻¹", 4*np.pi**3 + np.pi**2 + np.pi, 137.035999084, "CODATA 2022"),
    ("α⁻¹ (E60)", 137 - 1/2052, 137.035999084, "CODATA 2022"),
    ("η/λ", (np.pi/57)/(1/18), 14*np.pi/19, "E34 vs forma declarada"),
    ("m_p/m_e", 1836.15267343, 1836.15, "CODATA 2022 (RG aprox)"),
    ("Ω_DM", (np.pi - np.pi/57)/57, 0.2647, "Planck 2018"),
    ("H0 (km/s/Mpc)", 299792.458 * np.log(18/17) / 1.616255e-35 * 3.241e-20,
    67.4, "Planck 2018"),
    ("Λ (m⁻²)", 3/(1.616255e-35)**2, 1.1056e-52, "Planck 2018"),
]

for nombre, val_rg, val_ref, fuente in checks:
    err_abs = val_rg - val_ref
    err_rel = err_abs / val_ref if val_ref != 0 else np.inf
    estado = "OK" if abs(err_rel) < 0.01 else "FALLA"
    print(f"{nombre:<18} RG = {val_rg:<18.6e}")
    print(f"{'':18} Ref = {val_ref:<18.6e} ({fuente})")
    print(f"{'':18} Error rel = {err_rel:+.3e} → {estado}")
    print()

# ------------------------------------------------------------
# 5. FIGURAS
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A — Distribución de 𝔇 bajo perturbación
ax = axes[0, 0]
for name, res, color in [("σ=10%", results_10, "#3498DB"),
                          ("σ=30%", results_30, "#F39C12"),
                          ("σ=50%", results_50, "#E74C3C")]:
    D_vals = extraer(res, "D")
    ax.hist(D_vals, bins=60, alpha=0.5, color=color, label=name, density=True)
ax.axvline(1.5, color="black", ls="--", lw=2, label="Valor canónico 𝔇=1.5")
ax.set_xlabel("𝔇")
ax.set_ylabel("Densidad")
ax.set_title("A — Sensibilidad de 𝔇 frente a perturbación de roles")
ax.legend()
ax.grid(alpha=0.3)

# B — Estabilidad de identidades
ax = axes[0, 1]
sigmas = [10, 30, 50]
for eid, color in [("E17", "#27AE60"), ("E18", "#E74C3C"),
                    ("E21", "#3498DB"), ("E22", "#9B59B6")]:
    errors = []
    for res in [results_10, results_30, results_50]:
        vals = extraer(res, eid)
        errors.append(np.mean(np.abs(vals)))
    ax.plot(sigmas, errors, "o-", color=color, lw=2, ms=8, label=eid)
ax.set_yscale("symlog", linthresh=1e-10)
ax.set_xlabel("Perturbación σ (%)")
ax.set_ylabel("|Error| medio")
ax.set_title("B — Estabilidad de identidades algebraicas")
ax.legend()
ax.grid(alpha=0.3)

# C — Coincidencia numérica de π
ax = axes[1, 0]
variantes_vals = np.array([v for v, _ in expr_variantes(5)])
ax.hist(variantes_vals, bins=80, color="#3498DB", alpha=0.7, edgecolor="white")
ax.axvline(target, color="#E74C3C", ls="--", lw=2,
           label=f"α⁻¹ = {target}")
ax.axvspan(target - 0.1, target + 0.1, alpha=0.3, color="#E74C3C",
           label="ventana ±0.1")
ax.set_xlabel("Valor de k·πⁿ")
ax.set_ylabel("Frecuencia")
ax.set_xlim(0, 200)
ax.set_title(f"C — Distribución de expresiones simples de π")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# D — Errores relativos contra CODATA
ax = axes[1, 1]
nombres_plot = []
errores_plot = []
for nombre, val_rg, val_ref, fuente in checks:
    if val_ref != 0:
        err_rel = (val_rg - val_ref) / val_ref
        nombres_plot.append(nombre)
        errores_plot.append(err_rel)

colors = ["#E74C3C" if abs(e) > 0.01 else "#27AE60" for e in errores_plot]
ax.barh(nombres_plot, errores_plot, color=colors, edgecolor="black")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Error relativo (RG vs dato externo)")
ax.set_xscale("symlog", linthresh=1e-3)
ax.set_title("D — Contraste RG vs CODATA/Planck")
ax.grid(alpha=0.3, axis="x")
ax.invert_yaxis()

plt.suptitle("RG-108 · Pre-Monte Carlo · Paisaje de postulados", fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig("pre_mc_postulados.png", dpi=140, bbox_inches="tight")
plt.show()

print("Guardado: pre_mc_postulados.png")

In [ ]:
# ============================================================
# RG-108 · Motor AA (Auto-Ajuste Adaptativo)
# Búsqueda de coeficientes ocultos y correcciones estructurales
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from itertools import product

np.random.seed(108)

# ------------------------------------------------------------
# 0. Targets (valores de referencia)
# ------------------------------------------------------------
TARGETS = {
    "alpha_inv": 137.035999084,
    "eta_over_lambda": 14*np.pi/19,        # valor declarado en E34
    "m_p_m_e": 1836.15267343,
    "Omega_DM": 0.2647,
    "H0_SI": 2.184e-18,                    # s^-1 (Planck)
    "H0_kms_Mpc": 67.4,
    "Lambda": 1.1056e-52,                  # m^-2 (Planck)
}

# Constantes base
pi = np.pi
e = np.e
l_P = 1.616255e-35
c_ms = 299792458.0
hbar = 1.054571817e-34

eta_RG = pi / 57
lambda_RG = 1.0 / 18
Phi_RG = 18.0 / 17
D_RG = 1.5

# ------------------------------------------------------------
# 1. MOTOR DE BÚSQUEDA DE COEFICIENTES
# ------------------------------------------------------------
# Dada una "forma" (función de algún parámetro), busca el valor
# del parámetro que más se acerca al target.

def search_coefficient(forma_fn, target, rango, n_puntos=100000):
    """
    forma_fn(x) -> valor
    Busca x en rango tal que forma_fn(x) ≈ target.
    """
    xs = np.linspace(rango[0], rango[1], n_puntos)
    vals = np.array([forma_fn(x) for x in xs])
    idx = np.argmin(np.abs(vals - target))
    return xs[idx], vals[idx]

def search_integer_coefficient(forma_fn, target, k_max=100):
    """Busca coeficiente entero k tal que k * forma_fn() ≈ target."""
    best_k, best_err = None, np.inf
    for k in range(-k_max, k_max + 1):
        if k == 0: continue
        val = k * forma_fn()
        err = abs(val - target) / abs(target)
        if err < best_err:
            best_err = err
            best_k = k
    return best_k, best_err

# ------------------------------------------------------------
# 2. BÚSQUEDA ESPECÍFICA POR ECUACIÓN
# ------------------------------------------------------------

print("=" * 72)
print("AA · BÚSQUEDA DE COEFICIENTES")
print("=" * 72)

# --- E34: η/λ ---
print("\n[E34] η/λ = ?·π/19")
print("-" * 60)
eta_over_lambda_real = eta_RG / lambda_RG
print(f"  Cálculo directo: η/λ = {eta_over_lambda_real:.6f}")
print(f"  Declarado en E34: 14π/19 = {14*pi/19:.6f}")
print(f"  ¿Existe k entero tal que kπ/19 ≈ η/λ?")
for k in range(1, 30):
    val = k * pi / 19
    err = abs(val - eta_over_lambda_real) / eta_over_lambda_real
    marca = " ← MATCH" if err < 1e-10 else ""
    if err < 0.01:
        print(f"    k={k:2d}   {k}π/19 = {val:.6f}   err={err:.2e}{marca}")

# --- E60: α⁻¹ = 137 - 1/n ---
print("\n[E60] α⁻¹ = 137 - 1/n, buscar n óptimo")
print("-" * 60)
target = TARGETS["alpha_inv"]
n_opt = 1.0 / (137.0 - target)
print(f"  n óptimo = 1/(137 - α⁻¹_CODATA) = {n_opt:.4f}")
print(f"  Declarado en E60: n = 2052")
print(f"  α⁻¹ con n=2052:  {137 - 1/2052:.9f}")
print(f"  α⁻¹ con n={n_opt:.0f}:    {137 - 1/n_opt:.9f}")
print(f"  Target CODATA:   {target:.9f}")
print(f"  ¿n óptimo es entero? {abs(n_opt - round(n_opt)) < 0.01}")

# --- E34 alternative: ¿η o λ distinto? ---
print("\n[E34 alt] ¿Si λ = 1/18, cuál η daría η/λ = 14π/19?")
print("-" * 60)
eta_needed = (14 * pi / 19) * lambda_RG
print(f"  η necesario = {eta_needed:.6f}")
print(f"  η declarado (π/57) = {eta_RG:.6f}")
print(f"  Ratio: {eta_needed/eta_RG:.4f}")
print(f"  ¿π/k con qué k? k = {pi/eta_needed:.4f}")
print(f"  ¿Existe k entero tal que η = π/k? k = {pi/eta_needed:.4f}")

print("\n[E34 alt] ¿Si η = π/57, cuál λ daría η/λ = 14π/19?")
lambda_needed = eta_RG / (14 * pi / 19)
print(f"  λ necesario = {lambda_needed:.6f}")
print(f"  λ declarado (1/18) = {lambda_RG:.6f}")
print(f"  ¿λ = 1/k con qué k? k = {1/lambda_needed:.4f}")

# --- Λ = 3/ℓ_P² ---
print("\n[E77] Λ = ?/ℓ_P²")
print("-" * 60)
Lambda_target = TARGETS["Lambda"]
coef_opt = Lambda_target * l_P**2
print(f"  Coeficiente óptimo: c = Λ_obs · ℓ_P² = {coef_opt:.6e}")
print(f"  Declarado: c = 3")
print(f"  Discrepancia: {3/coef_opt:.3e}")

# ¿Cuál es la escala natural?
print("\n[E77 alt] ¿Existe otra escala natural?")
print(f"  Λ_obs · ℓ_P² = {coef_opt:.6e}")
print(f"  ¿(ℓ_EW/ℓ_P)²? ℓ_EW ~ 10⁻¹⁸ m: (1e-18/1.6e-35)² ≈ {(1e-18/l_P)**2:.3e}")
print(f"  ¿1/(M_P/M_EW)²? M_EW ~ 246 GeV: (246/1.22e19)² ≈ {(246/1.22e19)**2:.3e}")

# --- H0 ---
print("\n[E81] H₀ = ?")
print("-" * 60)
H0_target_SI = TARGETS["H0_SI"]
print(f"  Target H₀ (SI) = {H0_target_SI:.6e} s⁻¹")
print(f"  RG declara: c·ln(Φ)/ℓ_P")
val_RG = c_ms * np.log(Phi_RG) / l_P
print(f"  c·ln(Φ)/ℓ_P = {val_RG:.6e} s⁻¹")
print(f"  Ratio RG/target = {val_RG/H0_target_SI:.6e}")

# ¿Qué escala da H0 correcto?
L_H0 = c_ms / H0_target_SI
print(f"  Longitud característica: c/H₀ = {L_H0:.6e} m")
print(f"  ≈ ¿1/H₀ en unidades naturales? Sí, por definición.")
print(f"  ¿Es esta longitud una escala natural? Compare con ℓ_P = {l_P:.3e}, "
      f"radio Hubble ~ 1.3e26 m")

# --- Ω_DM ---
print("\n[E78] Ω_DM = (π-η)/57")
print("-" * 60)
Om_RG = (pi - eta_RG) / 57
print(f"  RG: (π - π/57)/57 = {Om_RG:.6f}")
print(f"  Target Planck: {TARGETS['Omega_DM']}")

# Búsqueda: Ω_DM = (π - η)/k ó (kπ - η)/m
print("\n  ¿Existe (π·a - η)/k ≈ Ω_DM con a,k enteros pequeños?")
best = None
for a in range(1, 10):
    for k in range(1, 100):
        val = (a*pi - eta_RG) / k
        err = abs(val - TARGETS["Omega_DM"])
        if best is None or err < best[3]:
            best = (a, k, val, err)
a, k, val, err = best
print(f"  Mejor: ({a}π - η)/{k} = {val:.6f}  (err={err:.2e})")

# ¿O simplemente π/k?
print("\n  ¿Existe π/k ó (π-η)/k con match exacto?")
for k in range(1, 100):
    for nombre, val in [("π/k", pi/k), ("(π-η)/k", (pi-eta_RG)/k)]:
        err = abs(val - TARGETS["Omega_DM"])
        if err < 0.005:
            print(f"    {nombre} con k={k}: {val:.6f}  err={err:.2e}")

# ------------------------------------------------------------
# 3. BARRIDO GENERAL: coeficiente óptimo por fórmula
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("AA · BARRIDO GENERAL")
print("=" * 72)

barridos = [
    ("E59", "α⁻¹ = 4π³+π²+π", 4*pi**3+pi**2+pi, TARGETS["alpha_inv"]),
    ("E60", "α⁻¹ = 137 - 1/2052", 137-1/2052, TARGETS["alpha_inv"]),
    ("E34", "η/λ = 14π/19", 14*pi/19, eta_RG/lambda_RG),
    ("E78", "Ω_DM = (π-η)/57", (pi-eta_RG)/57, TARGETS["Omega_DM"]),
    ("E81", "H₀ = c·ln(Φ)/ℓ_P", c_ms*np.log(Phi_RG)/l_P, TARGETS["H0_SI"]),
    ("E77", "Λ = 3/ℓ_P²", 3/l_P**2, TARGETS["Lambda"]),
]

print(f"{'ID':<6} {'Declarado':<18} {'Ratio decl/ref':<20} {'Estado':<10}")
print("-" * 72)
for eid, desc, val, ref in barridos:
    ratio = val / ref
    err_rel = abs(val - ref) / abs(ref)
    estado = "OK" if err_rel < 0.01 else ("TYPO?" if err_rel < 10 else "FALLA")
    print(f"{eid:<6} {desc[:17]:<18} {ratio:<20.6e} {estado:<10}")

# ------------------------------------------------------------
# 4. FIGURA
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# A — Búsqueda de k en E34: η/λ = kπ/19
ax = axes[0, 0]
eta_over_lambda_real = eta_RG / lambda_RG
ks = np.arange(1, 30)
vals = ks * pi / 19
ax.plot(ks, vals, "o-", color="#3498DB", lw=2, ms=8, label="kπ/19")
ax.axhline(eta_over_lambda_real, color="#27AE60", ls="--", lw=2,
           label=f"η/λ real = {eta_over_lambda_real:.4f}")
ax.axhline(14*pi/19, color="#E74C3C", ls=":", lw=2,
           label=f"E34 declarado = {14*pi/19:.4f}")
# Marcar mejor match
best_k = np.argmin(np.abs(vals - eta_over_lambda_real)) + 1
ax.scatter([best_k], [best_k*pi/19], s=200, facecolors="none",
           edgecolors="black", lw=2, zorder=5,
           label=f"Mejor match: k={best_k}")
ax.set_xlabel("k (entero)")
ax.set_ylabel("kπ/19")
ax.set_title("A — E34: buscar k óptimo en η/λ = kπ/19")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# B — E60: búsqueda de n
ax = axes[0, 1]
ns = np.arange(100, 5000)
alpha_vals = 137 - 1/ns
ax.plot(ns, alpha_vals, color="#3498DB", lw=2)
ax.axhline(TARGETS["alpha_inv"], color="#E74C3C", ls="--", lw=2,
           label=f"CODATA = {TARGETS['alpha_inv']:.6f}")
ax.axhline(137-1/2052, color="#F39C12", ls=":", lw=2,
           label="E60 declara n=2052")
# n óptimo
n_opt = 1.0 / (137 - TARGETS["alpha_inv"])
ax.axvline(n_opt, color="#27AE60", ls=":", lw=2,
           label=f"n óptimo = {n_opt:.1f}")
ax.set_xlabel("n")
ax.set_ylabel("137 - 1/n")
ax.set_title("B — E60: buscar n óptimo")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# C — E78: malla de búsqueda (a·π - η)/k
ax = axes[1, 0]
Omega_target = TARGETS["Omega_DM"]
malla = np.zeros((20, 100))
for i, a in enumerate(range(1, 21)):
    for j, k in enumerate(range(1, 101)):
        malla[i, j] = (a*pi - eta_RG) / k
im = ax.imshow(malla, aspect="auto", origin="lower",
               extent=[0.5, 100.5, 0.5, 20.5],
               cmap="viridis")
plt.colorbar(im, ax=ax, label="(aπ-η)/k")
# Marcar target
a_target_idx = np.argmin(np.abs(malla - Omega_target))
a_hit, k_hit = np.unravel_index(a_target_idx, malla.shape)
ax.scatter([k_hit+1], [a_hit+1], s=200, facecolors="none",
           edgecolors="red", lw=2.5, label=f"Match: ({a_hit+1}π-η)/{k_hit+1}")
ax.set_xlabel("k")
ax.set_ylabel("a")
ax.set_title(f"C — E78: búsqueda (aπ-η)/k ≈ {Omega_target}")
ax.legend(fontsize=8)

# D — Ratios de error por ecuación
ax = axes[1, 1]
ids = [b[0] for b in barridos]
errores = [abs(b[2] - b[3]) / abs(b[3]) for b in barridos]
colors = ["#27AE60" if e < 0.01 else ("#F39C12" if e < 10 else "#E74C3C")
          for e in errores]
ax.bar(ids, errores, color=colors, edgecolor="black")
ax.set_yscale("log")
ax.axhline(0.01, color="green", ls="--", lw=1, alpha=0.5, label="Umbral OK (1%)")
ax.axhline(10, color="red", ls="--", lw=1, alpha=0.5, label="Umbral TYPO (10×)")
ax.set_ylabel("Error relativo")
ax.set_title("D — Triaje: error relativo por ecuación")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis="y")

plt.suptitle("RG-108 · Motor AA · Búsqueda de valores", fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig("AA_busqueda_valores.png", dpi=140, bbox_inches="tight")
plt.show()

print("\nGuardado: AA_busqueda_valores.png")

# ------------------------------------------------------------
# 5. Reporte estructurado
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("AA · REPORTE ESTRUCTURADO")
print("=" * 72)

reporte = {
    "E34_eta_over_lambda": {
        "declarado": "14π/19",
        "calculado_directo": eta_RG/lambda_RG,
        "match_entero": best_k,
        "nuevo_coef_sugerido": f"{best_k}π/19",
        "veredicto": "TYPO_PROBABLE" if best_k != 14 else "OK",
    },
    "E60_alpha_inv": {
        "declarado_n": 2052,
        "n_optimo": float(n_opt),
        "es_entero": bool(abs(n_opt - round(n_opt)) < 0.5),
        "veredicto": "TYPO_O_INCOMPLETO",
    },
    "E77_Lambda": {
        "ratio": 3 / coef_opt,
        "ordenes_magnitud_off": float(np.log10(3/coef_opt)),
        "veredicto": "ESTRUCTURAL_NO_TYPO",
    },
    "E78_Omega_DM": {
        "mejor_forma": f"({a}π-η)/{k}",
        "match_valor": float(val),
        "veredicto": "REVISAR_ESTRUCTURA",
    },
    "E81_H0": {
        "ratio": val_RG/H0_target_SI,
        "veredicto": "ESTRUCTURAL_NO_TYPO",
    },
}

import json
print(json.dumps(reporte, indent=2, default=str))

In [ ]:
# ============================================================
# RG-108 · TEST DE SIGNIFICANCIA DE E59
# ¿α⁻¹ = 4π³ + π² + π es hallazgo o coincidencia?
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from itertools import product

np.random.seed(108)

# ------------------------------------------------------------
# 1. Constantes
# ------------------------------------------------------------
pi = np.pi
alpha_inv_CODATA = 137.035999084
alpha_inv_err_CODATA = 2.1e-8  # incertidumbre de CODATA

# E59 declarado
E59_value = 4*pi**3 + pi**2 + pi
E59_err_abs = abs(E59_value - alpha_inv_CODATA)
E59_err_rel = E59_err_abs / alpha_inv_CODATA

print("=" * 72)
print("TEST DE SIGNIFICANCIA · E59")
print("=" * 72)
print()
print(f"E59 declarado:     α⁻¹ = 4π³ + π² + π")
print(f"  Valor calculado: {E59_value:.12f}")
print(f"  CODATA 2022:     {alpha_inv_CODATA:.12f}")
print(f"  Error absoluto:  {E59_err_abs:.3e}")
print(f"  Error relativo:  {E59_err_rel:.3e}")
print()

# ------------------------------------------------------------
# 2. Barrido completo del espacio aπ³ + bπ² + cπ + d
# ------------------------------------------------------------
# Rango: [-N, N] en cada coeficiente.
# N=20 → 41^4 ≈ 2.8M expresiones. Factible.

N_max = 20
coefs = np.arange(-N_max, N_max + 1)

# Potencias base
pi3, pi2, pi1 = pi**3, pi**2, pi

print(f"Generando expresiones con a,b,c,d ∈ [{-N_max}, {N_max}]...")
print(f"  Total a generar: {len(coefs)}^4 = {len(coefs)**4:,}")
print()

# Generar vectorizado (sin loop explícito)
A = coefs.reshape(-1, 1, 1, 1)
B = coefs.reshape(1, -1, 1, 1)
C = coefs.reshape(1, 1, -1, 1)
D = coefs.reshape(1, 1, 1, -1)

valores = (A * pi3 + B * pi2 + C * pi1 + D).flatten()
# Filtrar valores en rango de interés (descarta la mayoría)
mask = (valores > 100) & (valores < 200)
valores_rango = valores[mask]

print(f"Expresiones totales:         {len(valores):,}")
print(f"Expresiones en [100, 200]:   {len(valores_rango):,}")
print()

# ------------------------------------------------------------
# 3. Contar matches por precisión
# ------------------------------------------------------------
print("=" * 72)
print("DISTRIBUCIÓN POR PRECISIÓN")
print("=" * 72)
print(f"{'Ventana ±ε':<15} {'Nº exprs':<15} {'Esperado (densidad)':<20} {'Significancia':<15}")
print("-" * 72)

# Densidad empírica: expresiones por unidad de valor
ancho_rango = 200 - 100
densidad = len(valores_rango) / ancho_rango
print(f"Densidad empírica: {densidad:.2f} expresiones / unidad de valor")
print()

resultados_precision = []
for epsilon in [1e0, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]:
    matches = np.sum(np.abs(valores_rango - alpha_inv_CODATA) < epsilon)
    esperado = densidad * 2 * epsilon
    if esperado > 0:
        significancia = matches / esperado
    else:
        significancia = np.inf if matches > 0 else 1.0
    resultados_precision.append((epsilon, matches, esperado, significancia))
    print(f"±{epsilon:<14.0e} {matches:<15} {esperado:<20.4f} {significancia:<15.4f}")

print()

# ------------------------------------------------------------
# 4. ¿Es E59 único en su precisión?
# ------------------------------------------------------------
print("=" * 72)
print("¿ES E59 ÚNICO EN SU PRECISIÓN?")
print("=" * 72)

# Buscar TODAS las expresiones a 10⁻⁵ del target
matches_cercanos = valores_rango[np.abs(valores_rango - alpha_inv_CODATA) < 1e-5]
print(f"Expresiones con |error| < 10⁻⁵: {len(matches_cercanos)}")
for v in matches_cercanos:
    print(f"  {v:.12f}   (error = {v - alpha_inv_CODATA:+.3e})")
print()

# Buscar a 10⁻⁶
matches_106 = valores_rango[np.abs(valores_rango - alpha_inv_CODATA) < 1e-6]
print(f"Expresiones con |error| < 10⁻⁶: {len(matches_106)}")
for v in matches_106:
    print(f"  {v:.12f}   (error = {v - alpha_inv_CODATA:+.3e})")
print()

# ------------------------------------------------------------
# 5. Encontrar expresiones alternativas con alta precisión
# ------------------------------------------------------------
print("=" * 72)
print("EXPRESIONES ALTERNATIVAS CERCANAS A α⁻¹")
print("=" * 72)

# Recuperar los coeficientes que generan cada match cercano
# Esto es lento; lo hacemos con búsqueda dirigida.
def valor(a, b, c, d):
    return a*pi**3 + b*pi**2 + c*pi + d

mejores = []
for a in range(-N_max, N_max+1):
    for b in range(-N_max, N_max+1):
        for c in range(-N_max, N_max+1):
            v_partial = a*pi3 + b*pi2 + c*pi1
            # d óptimo (redondeo)
            d_opt = round(alpha_inv_CODATA - v_partial)
            if abs(d_opt) > N_max:
                continue
            v_total = v_partial + d_opt
            err = abs(v_total - alpha_inv_CODATA)
            if err < 1e-5:
                mejores.append((a, b, c, d_opt, v_total, err))

mejores.sort(key=lambda x: x[5])

print(f"Top 20 expresiones con coeficientes enteros pequeños:")
print(f"{'(a,b,c,d)':<20} {'Valor':<18} {'Error':<15}")
print("-" * 72)
for a, b, c, d, v, err in mejores[:20]:
    marker = " ← E59" if (a,b,c,d) == (4,1,1,0) else ""
    print(f"{str((a,b,c,d)):<20} {v:<18.12f} {err:<15.3e}{marker}")

print()

# ¿Cuál es el ranking de E59?
rank_E59 = next((i for i, m in enumerate(mejores) if m[:4] == (4,1,1,0)), None)
print(f"Ranking de E59 entre las {len(mejores)} expresiones con |err| < 10⁻⁵: "
      f"#{rank_E59 + 1 if rank_E59 is not None else '>'+str(len(mejores))}")

# ------------------------------------------------------------
# 6. Distribución de errores: histograma
# ------------------------------------------------------------
print()
print("=" * 72)
print("DISTRIBUCIÓN DE ERRORES |expr − α⁻¹_CODATA|")
print("=" * 72)

errores = np.abs(valores_rango - alpha_inv_CODATA)
for umbral in [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]:
    pct = np.sum(errores < umbral) / len(errores) * 100
    print(f"  |err| < {umbral:.0e}: {pct:.4f}%  "
          f"({np.sum(errores < umbral)} expresiones)")

# ------------------------------------------------------------
# 7. FIGURA
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# A — Histograma global
ax = axes[0, 0]
ax.hist(valores_rango, bins=200, color="#3498DB", alpha=0.7, edgecolor="white")
ax.axvline(alpha_inv_CODATA, color="#E74C3C", ls="--", lw=2,
           label=f"α⁻¹_CODATA = {alpha_inv_CODATA:.4f}")
ax.set_xlabel("Valor de aπ³ + bπ² + cπ + d")
ax.set_ylabel("Frecuencia")
ax.set_title(f"A — Distribución de {len(valores_rango):,} expresiones en [100, 200]")
ax.legend()
ax.grid(alpha=0.3)

# B — Zoom alrededor de α⁻¹
ax = axes[0, 1]
ventana = valores_rango[np.abs(valores_rango - alpha_inv_CODATA) < 0.1]
ax.hist(ventana, bins=100, color="#27AE60", alpha=0.75, edgecolor="white")
ax.axvline(alpha_inv_CODATA, color="#E74C3C", ls="--", lw=2,
           label=f"CODATA = {alpha_inv_CODATA:.9f}")
ax.axvline(E59_value, color="#9B59B6", ls=":", lw=2,
           label=f"E59 = {E59_value:.9f}")
ax.set_xlabel("Valor")
ax.set_ylabel("Frecuencia")
ax.set_title(f"B — Zoom ±0.1 ({len(ventana)} expresiones)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# C — Densidad de matches por precisión
ax = axes[1, 0]
epsilons = [r[0] for r in resultados_precision]
matches = [r[1] for r in resultados_precision]
esperado = [r[2] for r in resultados_precision]
ax.plot(epsilons, matches, "o-", color="#E74C3C", lw=2, ms=8,
        label="Observado")
ax.plot(epsilons, esperado, "s--", color="#3498DB", lw=2, ms=8,
        label="Esperado (uniforme)")
ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=0.01)
ax.set_xlabel("Ventana ε")
ax.set_ylabel("Nº expresiones en ±ε")
ax.set_title("C — Observado vs Esperado por densidad")
ax.legend()
ax.grid(alpha=0.3, which="both")
ax.invert_xaxis()

# D — Ranking de las mejores expresiones
ax = axes[1, 1]
top_n = min(20, len(mejores))
coefs_top = [f"({m[0]},{m[1]},{m[2]},{m[3]})" for m in mejores[:top_n]]
errs_top = [m[5] for m in mejores[:top_n]]
colors_top = ["#E74C3C" if c == "(4,1,1,0)" else "#3498DB" for c in coefs_top]
ax.barh(range(top_n), errs_top, color=colors_top, edgecolor="black")
ax.set_yticks(range(top_n))
ax.set_yticklabels(coefs_top, fontsize=8)
ax.set_xlabel("Error absoluto vs α⁻¹_CODATA")
ax.set_xscale("log")
ax.set_title(f"D — Top {top_n} expresiones (rojo = E59)")
ax.grid(alpha=0.3, axis="x")
ax.invert_yaxis()

plt.suptitle("RG-108 · Significancia de E59: α⁻¹ = 4π³ + π² + π",
             fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig("E59_significancia.png", dpi=140, bbox_inches="tight")
plt.show()

print("Guardado: E59_significancia.png")

# ------------------------------------------------------------
# 8. VEREDICTO AUTOMÁTICO
# ------------------------------------------------------------
print()
print("=" * 72)
print("VEREDICTO AUTOMÁTICO")
print("=" * 72)

n_106 = len(matches_106)
n_105 = len(matches_cercanos)

if n_106 <= 3:
    veredicto = "HALLAZGO_GENUINO"
    interpretacion = (f"Solo {n_106} expresiones (de {len(valores_rango):,}) "
                      f"alcanzan precisión 10⁻⁶. E59 es altamente no trivial.")
elif n_106 <= 20:
    veredicto = "HALLAZGO_PARCIAL"
    interpretacion = (f"{n_106} expresiones alcanzan 10⁻⁶. E59 no es único "
                      f"pero sigue siendo raro.")
else:
    veredicto = "COINCIDENCIA_ESTADISTICA"
    interpretacion = (f"{n_106} expresiones alcanzan 10⁻⁶. El espacio es "
                      f"denso; E59 no es especial.")

print(f"  Expresiones con |error| < 10⁻⁵: {n_105}")
print(f"  Expresiones con |error| < 10⁻⁶: {n_106}")
print(f"  Veredicto: {veredicto}")
print(f"  {interpretacion}")
print()

In [ ]:

# ============================================================
# RG-108 · E59 · RERUN con filtro corregido
# ============================================================

import numpy as np

pi = np.pi
alpha_inv_CODATA = 137.035999084
E59_value = 4*pi**3 + pi**2 + pi
E59_err = abs(E59_value - alpha_inv_CODATA)

print("=" * 72)
print("RERUN · E59 con filtro corregido")
print("=" * 72)
print(f"E59 error real: {E59_err:.4e}")
print(f"Filtro de búsqueda: |error| < 1e-3 (nivel de E59 o mejor)")
print()

# Barrido dirigido con filtro corregido
N_max = 20
coefs = np.arange(-N_max, N_max + 1)
pi3, pi2, pi1 = pi**3, pi**2, pi

mejores = []
for a in range(-N_max, N_max+1):
    for b in range(-N_max, N_max+1):
        for c in range(-N_max, N_max+1):
            v_partial = a*pi3 + b*pi2 + c*pi1
            d_opt = round(alpha_inv_CODATA - v_partial)
            if abs(d_opt) > N_max:
                continue
            v_total = v_partial + d_opt
            err = abs(v_total - alpha_inv_CODATA)
            if err < 1e-3:
                # Complejidad: suma de valores absolutos de coeficientes
                complejidad = abs(a) + abs(b) + abs(c) + abs(d_opt)
                mejores.append({
                    "coefs": (a, b, c, d_opt),
                    "valor": v_total,
                    "error": err,
                    "complejidad": complejidad,
                })

mejores.sort(key=lambda x: x["error"])

print(f"Total de expresiones con |error| < 1e-3: {len(mejores)}")
print()
print(f"{'#':<4} {'(a,b,c,d)':<20} {'Valor':<20} {'Error':<12} {'Complejidad':<12}")
print("-" * 72)
for i, m in enumerate(mejores, 1):
    marker = " ← E59" if m["coefs"] == (4, 1, 1, 0) else ""
    print(f"{i:<4} {str(m['coefs']):<20} {m['valor']:<20.12f} "
          f"{m['error']:<12.3e} {m['complejidad']:<12}{marker}")

# Ahora ordenar por complejidad (simplicidad)
print()
print("=" * 72)
print("ORDENADAS POR SIMPLICIDAD (menor suma de coeficientes)")
print("=" * 72)
por_simplicidad = sorted(mejores, key=lambda x: (x["complejidad"], x["error"]))
print(f"{'#':<4} {'(a,b,c,d)':<20} {'Complejidad':<12} {'Error':<12}")
print("-" * 72)
for i, m in enumerate(por_simplicidad[:15], 1):
    marker = " ← E59" if m["coefs"] == (4, 1, 1, 0) else ""
    print(f"{i:<4} {str(m['coefs']):<20} {m['complejidad']:<12} "
          f"{m['error']:<12.3e}{marker}")

# Análisis
print()
print("=" * 72)
print("ANÁLISIS")
print("=" * 72)

rank_error = next(i for i, m in enumerate(mejores, 1)
                  if m["coefs"] == (4, 1, 1, 0))
rank_simple = next(i for i, m in enumerate(por_simplicidad, 1)
                   if m["coefs"] == (4, 1, 1, 0))

print(f"E59 (4,1,1,0):")
print(f"  Ranking por error:       #{rank_error} de {len(mejores)}")
print(f"  Ranking por simplicidad: #{rank_simple} de {len(mejores)}")
print()

# Comparar con el mejor de todos
mejor_error = mejores[0]
mejor_simple = por_simplicidad[0]

print(f"Mejor por error: {mejor_error['coefs']} "
      f"(error = {mejor_error['error']:.3e})")
print(f"Mejor por simplicidad: {mejor_simple['coefs']} "
      f"(complejidad = {mejor_simple['complejidad']})")
print()

# Veredicto honesto
if rank_error <= 3 and rank_simple <= 3:
    veredicto = "E59 destacado en ambas metricas"
elif rank_error <= 3:
    veredicto = "E59 preciso pero no el mas simple"
elif rank_simple <= 3:
    veredicto = "E59 simple pero no el mas preciso"
else:
    veredicto = "E59 no destaca en ninguna metrica"

print(f"Veredicto: {veredicto}")

In [ ]:

# ============================================================
# RG-108 · BATCH E1-E5 · Tests estilo E59
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

np.random.seed(108)

print("=" * 78)
print("BATCH E1-E5 · TESTS DE SIGNIFICANCIA ESTRUCTURAL")
print("=" * 78)
print()

# ============================================================
# E1 — Ĥ|0⟩ = 0 : ¿es alcanzable? ¿es restrictiva?
# ============================================================
print("─" * 78)
print("E1 — Ĥ|0⟩ = 0")
print("─" * 78)

N = 10
n_h = 5000

def test_E1(n_h=5000, N=10):
    """Para Hamiltonianos aleatorios hermitianos, mide el ground state.
    Luego aplica shift y vuelve a medir."""
    e_min_sin_shift = np.zeros(n_h)
    e_min_con_shift = np.zeros(n_h)
    for i in range(n_h):
        M = np.random.randn(N, N) + 1j * np.random.randn(N, N)
        H = (M + M.conj().T) / 2
        evals = np.linalg.eigvalsh(H)
        e_min_sin_shift[i] = evals[0]
        # shift
        H_shifted = H - evals[0] * np.eye(N)
        e_min_con_shift[i] = np.linalg.eigvalsh(H_shifted)[0]
    return e_min_sin_shift, e_min_con_shift

e_sin, e_con = test_E1(n_h, N)

# ¿Cuántos tienen E_min = 0 sin shift?
frac_exact_0_sin = np.mean(np.abs(e_sin) < 1e-10)
frac_exact_0_con = np.mean(np.abs(e_con) < 1e-10)

print(f"  N = {N}, n_h = {n_h} Hamiltonianos aleatorios")
print(f"  Fracción con E_min = 0 sin shift: {frac_exact_0_sin:.6f}")
print(f"  Fracción con E_min = 0 con shift: {frac_exact_0_con:.6f}")
print(f"  E_min medio sin shift: {e_sin.mean():.6f} ± {e_sin.std():.6f}")
print(f"  E_min medio con shift: {e_con.mean():.6f} ± {e_con.std():.6f}")
print()

veredicto_E1 = (
    "TRIVIAL (alcanzable por shift)" if frac_exact_0_con > 0.99
    else "RESTRICTIVO"
)
print(f"  Veredicto: {veredicto_E1}")
print()

# ============================================================
# E2 — ⟨0|0⟩ = 1 : ¿restricción o convención?
# ============================================================
print("─" * 78)
print("E2 — ⟨0|0⟩ = 1")
print("─" * 78)

def test_E2(n_samples=10000, N=10):
    """Vectores aleatorios en C^N. Fracción con norma 1 exactamente."""
    normas_antes = np.zeros(n_samples)
    normas_despues = np.zeros(n_samples)
    for i in range(n_samples):
        v = np.random.randn(N) + 1j * np.random.randn(N)
        normas_antes[i] = np.linalg.norm(v)
        v_norm = v / np.linalg.norm(v)
        normas_despues[i] = np.linalg.norm(v_norm)
    return normas_antes, normas_despues

n_antes, n_despues = test_E2(10000, N)

frac_norma_1_antes = np.mean(np.abs(n_antes - 1) < 1e-10)
frac_norma_1_despues = np.mean(np.abs(n_despues - 1) < 1e-10)

print(f"  Vectores aleatorios: {10000}, N = {N}")
print(f"  Fracción con ‖v‖ = 1 antes de reescalar:  {frac_norma_1_antes:.6f}")
print(f"  Fracción con ‖v‖ = 1 después de reescalar: {frac_norma_1_despues:.6f}")
print()

veredicto_E2 = (
    "CONVENCIÓN (siempre alcanzable)" if frac_norma_1_despues > 0.99
    else "RESTRICTIVO"
)
print(f"  Veredicto: {veredicto_E2}")
print()

# ============================================================
# E3 — [Ĥ, P̂(0)]|0⟩ = 0 : ¿se sigue de E1?
# ============================================================
print("─" * 78)
print("E3 — [Ĥ, P̂(0)]|0⟩ = 0 : ¿se sigue de E1?")
print("─" * 78)

def test_E3(n_samples=5000, N=6):
    """Para cada par (Ĥ, P̂) con Ĥ|0⟩=0, verifica si [Ĥ,P̂]|0⟩ = 0.
    El estado |0⟩ es el autovector de energía mínima."""
    frac_compatibles = 0
    magnitudes = []
    for i in range(n_samples):
        # Hamiltoniano aleatorio hermitiano
        M = np.random.randn(N, N) + 1j * np.random.randn(N, N)
        H = (M + M.conj().T) / 2
        evals, evecs = np.linalg.eigh(H)
        ket0 = evecs[:, 0]
        # Operador P̂ aleatorio hermitiano
        P = np.random.randn(N, N) + 1j * np.random.randn(N, N)
        P = (P + P.conj().T) / 2
        # [Ĥ, P̂]|0⟩ = Ĥ P̂|0⟩ − P̂ Ĥ|0⟩ = Ĥ P̂|0⟩ (porque Ĥ|0⟩ = 0)
        comm_result = H @ P @ ket0 - P @ H @ ket0
        magnitud = np.linalg.norm(comm_result)
        magnitudes.append(magnitud)
        if magnitud < 1e-10:
            frac_compatibles += 1
    return np.array(magnitudes), frac_compatibles / n_samples

magnitudes_E3, frac_E3 = test_E3(5000, 6)

print(f"  Pares (Ĥ, P̂) aleatorios: 5000")
print(f"  Fracción con [Ĥ,P̂]|0⟩ = 0: {frac_E3:.6f}")
print(f"  ‖[Ĥ,P̂]|0⟩‖ medio: {magnitudes_E3.mean():.6f}")
print(f"  ‖[Ĥ,P̂]|0⟩‖ mediana: {np.median(magnitudes_E3):.6f}")
print()

# ¿Cuándo SE CUMPLE E3 dado E1?
# E3 ⟺ P̂|0⟩ ∈ ker(Ĥ) ó P̂|0⟩ = 0
# Veamos: probar con P̂ = |0⟩⟨0| + algo

def test_E3_restrictivo(n_samples=1000, N=6):
    """Casos especiales: P̂ = proyector sobre ker(H), P̂ = proyector ortogonal."""
    frac = 0
    for i in range(n_samples):
        M = np.random.randn(N, N) + 1j * np.random.randn(N, N)
        H = (M + M.conj().T) / 2
        evals, evecs = np.linalg.eigh(H)
        ket0 = evecs[:, 0]
        # P̂ = proyector sobre |0⟩
        P = np.outer(ket0, ket0.conj())
        comm_result = H @ P @ ket0 - P @ H @ ket0
        if np.linalg.norm(comm_result) < 1e-10:
            frac += 1
    return frac / n_samples

frac_especial = test_E3_restrictivo(1000, 6)
print(f"  Fracción si P̂ = |0⟩⟨0|: {frac_especial:.6f}")
print()

veredicto_E3 = (
    "SE SIGUE DE E1" if frac_E3 > 0.9
    else "NO SE SIGUE, REQUIERE P̂ ESPECIAL"
)
print(f"  Veredicto: {veredicto_E3}")
print()

# ============================================================
# E4 — Ĥ = ΣJ[1−cos(Δψ)] : ¿ground state 0? ¿espectro real?
# ============================================================
print("─" * 78)
print("E4 — Ĥ = ΣJ[1−cos(Δψ)]")
print("─" * 78)

def test_E4(n_redes=1000, N_nodos=6, J=1.0):
    """Red aleatoria con N nodos. Calcula Ĥ = Σ_{edges} J[1-cos(ψ_i - ψ_j)].
    En el punto fijo ψ_i = 0 para todos, E = 0. Verifica E_min."""
    E_en_cero = np.zeros(n_redes)
    E_min_random = np.zeros(n_redes)
    for i in range(n_redes):
        # Red aleatoria: matriz de adyacencia
        A = (np.random.rand(N_nodos, N_nodos) < 0.4).astype(float)
        A = np.triu(A, 1)
        A = A + A.T
        # Fases aleatorias
        psi = np.random.uniform(-np.pi, np.pi, N_nodos)
        # Ĥ en ψ: suma sobre aristas
        E = 0.0
        for j in range(N_nodos):
            for k in range(j+1, N_nodos):
                if A[j, k] > 0:
                    E += J * (1 - np.cos(psi[j] - psi[k]))
        E_min_random[i] = E
        # E con ψ_i = 0 para todos:
        E_en_cero[i] = 0.0
    return E_en_cero, E_min_random

E_cero, E_rand = test_E4(1000, 6)

print(f"  Redes aleatorias: 1000, N nodos = 6, J = 1")
print(f"  E en ψ_i = 0: {E_cero.mean():.6e} ± {E_cero.std():.6e}")
print(f"  E con ψ aleatorio: {E_rand.mean():.6f} ± {E_rand.std():.6f}")
print(f"  Fracción con E(ψ=0) = 0 exacto: {np.mean(np.abs(E_cero) < 1e-15):.6f}")
print()

# ¿Cuándo E < 0? Nunca, porque 1 - cos(θ) ≥ 0
E_min_posible = 0.0
print(f"  Como 1 − cos(θ) ≥ 0 para todo θ real, Ĥ ≥ 0 siempre.")
print(f"  Ground state = 0 alcanzable con ψ uniforme.")
print()

veredicto_E4 = "ESTRUCTURAL (ground state 0 garantizado por forma coseno)"
print(f"  Veredicto: {veredicto_E4}")
print()

# ============================================================
# E5 — φ(−k) = −φ*(k) : ¿restricción fuerte?
# ============================================================
print("─" * 78)
print("E5 — φ(−k) = −φ*(k)")
print("─" * 78)

def test_E5(n_samples=10000, N_k=50):
    """Función compleja aleatoria en N_k puntos. Verifica la condición
    φ(-k) = -φ*(k) y mide la fracción que la cumple."""
    frac_cumple = 0
    magnitudes = []
    for i in range(n_samples):
        # φ en puntos k = -N_k/2, ..., N_k/2 (simétrico)
        k_vals = np.arange(-N_k//2, N_k//2)
        phi = np.random.randn(len(k_vals)) + 1j * np.random.randn(len(k_vals))
        # Condición: φ(-k) = -φ*(k)
        # Emparejar: para cada k, -k
        cumple = True
        for j, k in enumerate(k_vals):
            k_neg_idx = np.where(k_vals == -k)[0]
            if len(k_neg_idx) == 0:
                continue
            if abs(phi[j] + np.conj(phi[k_neg_idx[0]])) > 1e-10:
                cumple = False
                break
        if cumple:
            frac_cumple += 1
    return frac_cumple / n_samples

frac_E5_random = test_E5(10000, 50)

# Ahora probemos con función real-odd en espacio directo
def test_E5_estructura(n_samples=1000, N_x=64):
    """Dado f real y odd en espacio directo, FFT da φ(k).
    Verifica si φ cumple φ(-k) = -φ*(k)."""
    frac = 0
    for i in range(n_samples):
        x = np.linspace(-10, 10, N_x)
        f = np.random.randn(N_x)  # aleatorio
        # Forzar real y odd
        f = np.real(f)
        f_odd = (f - f[::-1]) / 2  # odd
        # FFT
        phi = np.fft.fft(f_odd)
        # Verificar condición en pares conjugados
        N = len(phi)
        k = np.arange(N)
        # Condición: φ(-k) = -φ*(k) → φ(N-k) = -conj(φ(k)) para k>0
        cond_ok = True
        for j in range(1, N//2):
            if abs(phi[N-j] + np.conj(phi[j])) > 1e-8:
                cond_ok = False
                break
        if cond_ok:
            frac += 1
    return frac / n_samples

frac_E5_estructura = test_E5_estructura(1000, 64)

print(f"  Fracción de funciones aleatorias que cumplen: {frac_E5_random:.6f}")
print(f"  Fracción si f es real-odd (por construcción): {frac_E5_estructura:.6f}")
print()

veredicto_E5 = (
    "IDENTIDAD ESTRUCTURAL (FT de real-odd)" if frac_E5_estructura > 0.99
    else "CONDICIÓN ADICIONAL"
)
print(f"  Veredicto: {veredicto_E5}")
print()

# ============================================================
# FIGURA RESUMEN
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# E1
ax = axes[0, 0]
ax.hist(e_sin, bins=50, color="#E74C3C", alpha=0.6, label="Sin shift", edgecolor="white")
ax.hist(e_con, bins=50, color="#27AE60", alpha=0.6, label="Con shift", edgecolor="white")
ax.axvline(0, color="black", ls="--", lw=2)
ax.set_xlabel("E_min")
ax.set_ylabel("Frecuencia")
ax.set_title("E1 — E_min con/sin shift")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# E2
ax = axes[0, 1]
ax.hist(n_antes, bins=50, color="#3498DB", alpha=0.6, label="Antes", edgecolor="white")
ax.hist(n_despues, bins=50, color="#9B59B6", alpha=0.6, label="Después", edgecolor="white")
ax.axvline(1, color="black", ls="--", lw=2)
ax.set_xlabel("‖v‖")
ax.set_ylabel("Frecuencia")
ax.set_title("E2 — Normalización")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# E3
ax = axes[0, 2]
ax.hist(np.log10(magnitudes_E3 + 1e-15), bins=50, color="#F39C12", alpha=0.75, edgecolor="white")
ax.axvline(np.log10(1e-10), color="#E74C3C", ls="--", lw=2, label="Umbral E3")
ax.set_xlabel("log₁₀ ‖[Ĥ,P̂]|0⟩‖")
ax.set_ylabel("Frecuencia")
ax.set_title(f"E3 — Fracción cumple = {frac_E3:.4f}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# E4
ax = axes[1, 0]
ax.hist(E_rand, bins=50, color="#9B59B6", alpha=0.75, edgecolor="white")
ax.axvline(0, color="black", ls="--", lw=2, label="E(ψ uniforme) = 0")
ax.set_xlabel("E (energía de red)")
ax.set_ylabel("Frecuencia")
ax.set_title("E4 — E con ψ aleatorio vs 0")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# E5
ax = axes[1, 1]
labels_E5 = ["Aleatorio", "Real-odd (FT)"]
vals_E5 = [frac_E5_random, frac_E5_estructura]
colors_E5 = ["#E74C3C" if v < 0.5 else "#27AE60" for v in vals_E5]
bars = ax.bar(labels_E5, vals_E5, color=colors_E5, edgecolor="black")
for bar, v in zip(bars, vals_E5):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.3f}",
            ha="center", fontsize=10)
ax.set_ylabel("Fracción que cumple")
ax.set_title("E5 — Fracción con φ(−k) = −φ*(k)")
ax.set_ylim(0, 1.1)
ax.grid(alpha=0.3, axis="y")

# Resumen
ax = axes[1, 2]
ax.axis("off")
resumen_text = f"""
RESUMEN BATCH E1-E5

E1: {veredicto_E1[:30]}
E2: {veredicto_E2[:30]}
E3: {veredicto_E3[:30]}
E4: {veredicto_E4[:30]}
E5: {veredicto_E5[:30]}
"""
ax.text(0.05, 0.95, resumen_text, transform=ax.transAxes,
        fontsize=10, va="top", family="monospace",
        bbox=dict(boxstyle="round", facecolor="#F4F4F4",
                  edgecolor="#333", linewidth=1.5))

plt.suptitle("RG-108 · Batch E1-E5 · Tests de significancia", fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig("batch_E1_E5.png", dpi=140, bbox_inches="tight")
plt.show()

print("Guardado: batch_E1_E5.png")
print()

# ============================================================
# RESUMEN FINAL
# ============================================================
print("=" * 78)
print("RESUMEN BATCH E1-E5")
print("=" * 78)
print(f"""
E1 · Ĥ|0⟩ = 0
  → {veredicto_E1}
  → Trivial por shift. No restrictivo. Ya cerrado como [POSTULADO].

E2 · ⟨0|0⟩ = 1
  → {veredicto_E2}
  → Convención de representación. Ya cerrado como [POSTULADO-CONVENCIÓN].

E3 · [Ĥ, P̂(0)]|0⟩ = 0
  → {veredicto_E3}
  → NO se sigue de E1. Requiere P̂|0⟩ ∈ ker(Ĥ) o P̂|0⟩ = 0.
  → Nuevo.

E4 · Ĥ = Σ J[1−cos(Δψ)]
  → {veredicto_E4}
  → Ground state 0 garantizado por estructura coseno.
  → Nuevo.

E5 · φ(−k) = −φ*(k)
  → {veredicto_E5}
  → Identidad estructural: es la condición exacta para que
    φ sea la FT de una función real y odd.
  → Nuevo.
""")

In [ ]:

# ============================================================
# E5 · TEST CORREGIDO
# φ(−k) = −φ*(k) caracteriza la FT de funciones imaginarias puras
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(108)

def verifica_E5_DFT(f, tol=1e-8):
    """Verifica si la DFT de f cumple φ(-k) = -conj(φ(k))."""
    phi = np.fft.fft(f)
    N = len(phi)
    for k in range(N):
        k_neg = (-k) % N
        if abs(phi[k_neg] - (-np.conj(phi[k]))) > tol:
            return False
    return True

# --- Tests por clase funcional ---

n_samples = 500
N_x = 64

print("=" * 72)
print("E5 · TEST CORREGIDO")
print(f"n_samples = {n_samples}, N_x = {N_x}")
print("=" * 72)
print()

# 1. f aleatoria compleja
frac_random = 0
for _ in range(n_samples):
    f = np.random.randn(N_x) + 1j * np.random.randn(N_x)
    if verifica_E5_DFT(f):
        frac_random += 1
frac_random /= n_samples
print(f"f aleatoria compleja:        {frac_random:.4f}")

# 2. f real
frac_real = 0
for _ in range(n_samples):
    f = np.random.randn(N_x)
    if verifica_E5_DFT(f):
        frac_real += 1
frac_real /= n_samples
print(f"f real:                      {frac_real:.4f}")

# 3. f real-odd
frac_real_odd = 0
for _ in range(n_samples):
    g = np.random.randn(N_x)
    f = np.zeros(N_x)
    for n in range(1, N_x):
        f[n] = g[n] - g[N_x - n]
    if verifica_E5_DFT(f):
        frac_real_odd += 1
frac_real_odd /= n_samples
print(f"f real-odd:                  {frac_real_odd:.4f}")

# 4. f imaginaria pura
frac_imag = 0
for _ in range(n_samples):
    g = np.random.randn(N_x)
    f = 1j * g
    if verifica_E5_DFT(f):
        frac_imag += 1
frac_imag /= n_samples
print(f"f imaginaria pura (i·g):     {frac_imag:.4f}")

# 5. f real-par
frac_real_even = 0
for _ in range(n_samples):
    g = np.random.randn(N_x)
    f = np.zeros(N_x)
    for n in range(1, N_x):
        f[n] = g[n] + g[N_x - n]
    if verifica_E5_DFT(f):
        frac_real_even += 1
frac_real_even /= n_samples
print(f"f real-par:                  {frac_real_even:.4f}")

# 6. f imaginaria-odd
frac_imag_odd = 0
for _ in range(n_samples):
    g = np.random.randn(N_x)
    f = np.zeros(N_x, dtype=complex)
    for n in range(1, N_x):
        f[n] = 1j * (g[n] - g[N_x - n])
    if verifica_E5_DFT(f):
        frac_imag_odd += 1
frac_imag_odd /= n_samples
print(f"f imaginaria-odd:            {frac_imag_odd:.4f}")

# 7. f imaginaria-par
frac_imag_even = 0
for _ in range(n_samples):
    g = np.random.randn(N_x)
    f = np.zeros(N_x, dtype=complex)
    for n in range(1, N_x):
        f[n] = 1j * (g[n] + g[N_x - n])
    if verifica_E5_DFT(f):
        frac_imag_even += 1
frac_imag_even /= n_samples
print(f"f imaginaria-par:            {frac_imag_even:.4f}")

print()

# ------------------------------------------------------------
# Verificación analítica: condición necesaria y suficiente
# ------------------------------------------------------------
print("=" * 72)
print("¿QUÉ CLASE CARACTERIZA E5 EXACTAMENTE?")
print("=" * 72)

# E5 en espacio k ⟺ f[n] puramente imaginaria en espacio directo
# Verificación: mostrar parte real de f debe ser ~0

g = np.random.randn(N_x)
f_imag = 1j * g
phi = np.fft.fft(f_imag)

# Reconstruir f desde phi
f_reconstruida = np.fft.ifft(phi)

print(f"f imaginaria pura: max|Re(f)| = {np.max(np.abs(np.real(f_reconstruida))):.2e}")
print(f"                   max|Im(f)| = {np.max(np.abs(np.imag(f_reconstruida))):.4f}")
print()

# Correlación recíproca
g = np.random.randn(N_x)
f_real = g.copy()
phi_real = np.fft.fft(f_real)
frac_cumple_E5 = verifica_E5_DFT(f_real)
print(f"f real: ¿cumple E5? {frac_cumple_E5}")
print(f"  Razón: si f real, φ hermitiana: φ(-k) = +conj(φ(k))")
print(f"  E5 exige: φ(-k) = -conj(φ(k)). Solo se cumple si φ ≡ 0.")
print()

# ------------------------------------------------------------
# Verificación de E5 con la corrección de "puramente imaginaria"
# ------------------------------------------------------------
print("=" * 72)
print("VEREDICTO CORREGIDO")
print("=" * 72)

resultados = {
    "Aleatoria compleja": frac_random,
    "Real": frac_real,
    "Real-odd": frac_real_odd,
    "Real-par": frac_real_even,
    "Imaginaria pura": frac_imag,
    "Imaginaria-odd": frac_imag_odd,
    "Imaginaria-par": frac_imag_even,
}

for nombre, frac in sorted(resultados.items(), key=lambda x: -x[1]):
    marca = ""
    if frac > 0.99:
        marca = " ← CUMPLE E5"
    elif frac < 0.01 and frac != frac_random:
        marca = " (falla por estructura)"
    print(f"  {nombre:<25} {frac:.4f}{marca}")

print()
print("Conclusión:")
print("  E5 ⟺ φ es la FT de una función puramente imaginaria.")
print("  Es una identidad estructural, no un postulado físico.")

# ------------------------------------------------------------
# FIGURA
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: fracción por clase
ax = axes[0]
nombres = list(resultados.keys())
valores = list(resultados.values())
colores = ["#E74C3C" if v < 0.5 else "#27AE60" for v in valores]
bars = ax.barh(nombres, valores, color=colores, edgecolor="black")
ax.axvline(0.5, color="black", ls="--", lw=1, alpha=0.5)
for bar, v in zip(bars, valores):
    ax.text(v + 0.02, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("Fracción que cumple φ(−k) = −φ*(k)")
ax.set_title("A — E5 por clase funcional")
ax.set_xlim(0, 1.1)
ax.grid(alpha=0.3, axis="x")

# Panel B: verificación directa
ax = axes[1]
g = np.random.randn(N_x)
f_imag = 1j * g
phi_imag = np.fft.fft(f_imag)
k = np.arange(N_x)
ax.plot(k, np.real(phi_imag), "o-", color="#3498DB", lw=2, ms=5,
        label="Re(φ(k))")
ax.plot(k, np.imag(phi_imag), "s-", color="#E74C3C", lw=2, ms=5,
        label="Im(φ(k))")
# Verificar E5: φ(-k) = -conj(φ(k))
phi_neg = phi_imag[(-k) % N_x]
neg_conj = -np.conj(phi_imag)
ax.plot(k, np.real(phi_neg), ":", color="#27AE60", lw=2,
        label="Re(φ(−k))")
ax.plot(k, np.real(neg_conj), "--", color="#F39C12", lw=2,
        label="Re(−φ*(k))")
ax.set_xlabel("k")
ax.set_ylabel("φ(k)")
ax.set_title("B — Verificación directa: φ(−k) = −φ*(k) para f imaginaria")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

plt.suptitle("E5 · Test corregido", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("E5_corregido.png", dpi=140, bbox_inches="tight")
plt.show()

print("Guardado: E5_corregido.png")

In [ ]:

# ============================================================
# RG-108 · BATCH E6-E10 · AA + Monte Carlo con estado E1-E5
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

np.random.seed(610)

# ------------------------------------------------------------
# 0. Estado previo (E1-E5 blindados)
# ------------------------------------------------------------
STATE_BASE = {
    "E1": "CONVENCION · Ĥ|0⟩=0 trivial por shift",
    "E2": "CONVENCION · ⟨0|0⟩=1 trivial por reescalado",
    "E3": "POSTULADO  · requiere P̂(0)|0⟩ ∈ ker(Ĥ)",
    "E4": "ESTRUCTURAL· ground state 0 por forma coseno",
    "E5": "IDENTIDAD  · φ = FT de función imaginaria pura",
}

print("=" * 76)
print("BATCH E6-E10 · AA + Monte Carlo")
print("=" * 76)
print("\nEstado previo blindado:")
for k, v in STATE_BASE.items():
    print(f"  {k}: {v}")
print()

# Constantes base heredadas
D_info = 1.5
pi = np.pi

# ============================================================
# E6 — V(φ) = (𝔇/N)[...]  · INCOMPLETA
# ============================================================
print("─" * 76)
print("E6 — V(φ) = (𝔇/N)[...]  · INCOMPLETA")
print("─" * 76)

# Sin fórmula completa, solo podemos probar la estructura general.
# Hipótesis: V(φ) es un potencial tipo doble pozo o metaestable.
# Suposición razonable: V(φ) = (𝔇/N) · f(φ) con f(φ) par, f(0)=0.
# Test: ¿qué forma de f(φ) es compatible con E4 (ground state en 0)?

def test_E6_estructura():
    """
    Probar distintas formas de f(φ) y ver si el ground state queda en 0.
    Requisito E4: V(0) = 0 y V(φ) ≥ 0 en vecindad.
    """
    phi = np.linspace(-3, 3, 500)
    formas = {
        "φ² (parabólico)": phi**2,
        "φ⁴ (cuártico)": phi**4,
        "1-cos(φ)": 1 - np.cos(phi),
        "φ²(1-φ²/2) (meta)": phi**2 * (1 - phi**2/2),
        "cos(φ)-1": np.cos(phi) - 1,
    }
    print(f"{'Forma':<30} {'V(0)':<12} {'min local?':<12} {'V≥0 cerca?':<12}")
    print("-" * 76)
    for nombre, V in formas.items():
        V0 = V[np.argmin(np.abs(phi))]
        es_min = V0 <= V.min() + 1e-10
        v_positivo = np.all(V[(phi > -0.5) & (phi < 0.5)] >= -1e-10)
        print(f"{nombre:<30} {V0:<12.6f} {str(es_min):<12} {str(v_positivo):<12}")

    # Metaestable: compatible con E4 si V(0)=0 y hay mínimo local
    print()
    print("Compatibilidad con E4 (ground state en 0):")
    print("  φ²:         SÍ, mínimo global en 0")
    print("  φ⁴:         SÍ, mínimo global en 0")
    print("  1-cos(φ):   SÍ, mínimo global en 0 (= E4)")

test_E6_estructura()
print()
print("VEREDICTO E6: [PENDIENTE] — falta fórmula completa.")
print("  Compatible con E4 si V(0)=0 y V≥0 en vecindad (parabólico, cuártico, coseno).")
print()

# ============================================================
# E7 — Ψ₀(θ) = 1/√Vol(𝕋ᴺ)  · AA sobre Vol
# ============================================================
print("─" * 76)
print("E7 — Ψ₀(θ) = 1/√Vol(𝕋ᴺ)")
print("─" * 76)

def test_E7_vol(N_dims=[1, 2, 3], L_vals=[1.0, 10.0, 100.0]):
    """
    En T^N de lado L, Vol = L^N.
    Ψ₀ = 1/√Vol.
    Test: ∫|Ψ₀|² dθ = 1 sobre T^N.
    """
    print(f"{'N':<4} {'L':<8} {'Vol':<14} {'Ψ₀':<14} {'∫|Ψ₀|² dθ':<14}")
    print("-" * 76)
    for N in N_dims:
        for L in L_vals:
            Vol = L**N
            Psi0 = 1 / np.sqrt(Vol)
            # Integral sobre [0,L]^N de |Ψ₀|² = |Ψ₀|² · Vol = 1
            integral = abs(Psi0)**2 * Vol
            print(f"{N:<4} {L:<8} {Vol:<14.4f} {Psi0:<14.4f} {integral:<14.6f}")
    print()
    print("Conclusión: E7 normaliza correctamente para cualquier Vol finito.")
    print("Pero si Vol → ∞, Ψ₀ → 0 (problema C-006).")

test_E7_vol()

# AA: ¿qué pasa si el espacio NO es toro?
# ¿Tiene que ser Vol(T^N)? Probar alternativas
print()
print("AA: ¿qué normalización alternativa serviría?")
alternativas = {
    "1/√Vol(T^N)": "Toroidal",
    "(2π)^(-N/2)": "Fourier estándar (Vol = (2π)^N)",
    "1/√(2π)^N": "idem",
    "N^(-1/2)": "Por número de modos",
}
for forma, sentido in alternativas.items():
    print(f"  {forma:<20} → {sentido}")
print()
print("VEREDICTO E7: [POSTULADO]")
print("  Compatible con E2 (norma=1).")
print("  Conflicto con E1 si Vol → ∞ (C-006 activo).")
print()

# ============================================================
# E8 — P̂(θ) = cos(θ)  · ¿operador o escalar?
# ============================================================
print("─" * 76)
print("E8 — P̂(θ) = cos(θ)")
print("─" * 76)

# Dos interpretaciones:
# A) Operador multiplicativo: (P̂ψ)(θ) = cos(θ)·ψ(θ)
# B) Operador de proyección abstracto

def test_E8_operador_mult(N_basis=200):
    """P̂ como operador multiplicativo en L²[0, 2π]."""
    theta = np.linspace(0, 2*np.pi, N_basis, endpoint=False)
    dtheta = 2*np.pi / N_basis
    # Diagonal: cos(θ_i)
    P_diag = np.cos(theta)
    # Espectro
    eigs = np.sort(np.unique(np.round(P_diag, 8)))
    print(f"  P̂ como multiplicador cos(θ) en L²[0,2π]:")
    print(f"    Espectro continuo: [-1, 1]")
    print(f"    Valores extremos: {P_diag.min():.4f}, {P_diag.max():.4f}")
    print(f"    ¿Hermitiano? Sí (real)")
    print()

def test_E8_proyector(N=8):
    """P̂ como proyector ortogonal en C^N."""
    # Proyector sobre un subespacio de dimensión 1
    v = np.random.randn(N) + 1j * np.random.randn(N)
    v /= np.linalg.norm(v)
    P = np.outer(v, v.conj())
    print(f"  P̂ como proyector en C^{N}:")
    print(f"    ¿P² = P? {np.allclose(P @ P, P)}")
    print(f"    ¿P† = P? {np.allclose(P.conj().T, P)}")
    print(f"    Autovalores: {np.sort(np.linalg.eigvalsh(P))}")
    print()

test_E8_operador_mult()
test_E8_proyector()

print("VEREDICTO E8: [PENDIENTE-AMBIGUO]")
print("  Si P̂(θ) = cos(θ) es operador multiplicativo → espectro continuo [-1,1]")
print("  Si es proyector abstracto → espectro {0,1}")
print("  Necesita aclaración: ¿qué espacio actúa? ¿cómo se relaciona con E3?")
print()

# E8 vs E3: si P̂(θ) = cos(θ) multiplicador, ¿P̂(0)|0⟩ ∈ ker(Ĥ)?
# P̂(0) = cos(0) = 1 (identidad)
print("  Chequeo con E3:")
print("    P̂(0) = cos(0) = 1 (si multiplicador)")
print("    Entonces E3 dice: [Ĥ, 𝕀]|0⟩ = 0 → trivialmente cierto.")
print("    Si P̂(0) = 𝕀, E3 se cumple automáticamente.")
print()

# ============================================================
# E9 — ⟨0|0⟩/cos(0) = 1  · test trivial por E2 + E8
# ============================================================
print("─" * 76)
print("E9 — ⟨0|0⟩/cos(0) = 1")
print("─" * 76)

# E2 dice ⟨0|0⟩ = 1
# E8 dice cos(0) = 1
# E9 dice 1/1 = 1 (identidad trivial)

norma = 1.0
cos0 = np.cos(0.0)
E9_lhs = norma / cos0
print(f"  ⟨0|0⟩ = {norma} (de E2)")
print(f"  cos(0) = {cos0} (de E8)")
print(f"  E9 LHS = {norma}/{cos0} = {E9_lhs}")
print(f"  ¿= 1? {np.isclose(E9_lhs, 1.0)}")
print()
print("VEREDICTO E9: [IDENTIDAD-TRIVIAL]")
print("  E9 no aporta contenido nuevo. Es consecuencia de E2 ∧ E8.")
print()

# ============================================================
# E10 — ℛ = |0⟩⟨0|  · test idempotencia (depende de E2)
# ============================================================
print("─" * 76)
print("E10 — ℛ = |0⟩⟨0|")
print("─" * 76)

def test_E10(N=6, n_samples=5000):
    """
    Si ⟨0|0⟩ = 1 (E2), ℛ es proyector: ℛ² = ℛ.
    Si ⟨0|0⟩ ≠ 1, ℛ NO es proyector.
    """
    frac_norm1 = 0
    frac_norm2 = 0
    frac_norm05 = 0
    for _ in range(n_samples):
        v = np.random.randn(N) + 1j * np.random.randn(N)
        v /= np.linalg.norm(v)  # norma 1

        # Caso 1: norma = 1 (E2 vigente)
        R1 = np.outer(v, v.conj())
        if np.allclose(R1 @ R1, R1):
            frac_norm1 += 1

        # Caso 2: norma = 2 (E2 violado)
        v2 = 2 * v
        R2 = np.outer(v2, v2.conj())
        if np.allclose(R2 @ R2, R2):
            frac_norm2 += 1

        # Caso 3: norma = 0.5 (E2 violado)
        v3 = 0.5 * v
        R3 = np.outer(v3, v3.conj())
        if np.allclose(R3 @ R3, R3):
            frac_norm05 += 1

    return frac_norm1/n_samples, frac_norm2/n_samples, frac_norm05/n_samples

f1, f2, f05 = test_E10(6, 5000)
print(f"  Con norma 1 (E2 vigente):   {f1:.6f} → idempotente")
print(f"  Con norma 2 (E2 violado):   {f2:.6f} → NO idempotente")
print(f"  Con norma 0.5 (E2 violado): {f05:.6f} → NO idempotente")
print()
print("VEREDICTO E10: [ESTRUCTURAL CON DEPENDENCIA]")
print("  ℛ² = ℛ ⟺ ⟨0|0⟩ = 1 (E2).")
print("  E10 está estructuralmente ligado a E2.")
print("  Confirmación numérica: E2 no es decorativa, HABILITA E10.")
print()

# ============================================================
# FIGURA RESUMEN
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# E6 — potenciales
ax = axes[0, 0]
phi = np.linspace(-3, 3, 500)
ax.plot(phi, phi**2, label="φ²", color="#3498DB")
ax.plot(phi, phi**4, label="φ⁴", color="#E74C3C")
ax.plot(phi, 1-np.cos(phi), label="1−cos(φ)", color="#27AE60")
ax.axhline(0, color="black", ls="--", lw=1)
ax.set_xlabel("φ")
ax.set_ylabel("V(φ)")
ax.set_title("E6 — Formas candidatas para V(φ)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# E7 — normalización
ax = axes[0, 1]
Ls = np.logspace(-1, 2, 100)
for N in [1, 2, 3]:
    Vol = Ls**N
    Psi0 = 1/np.sqrt(Vol)
    ax.plot(Ls, Psi0, lw=2, label=f"N={N}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("L (lado del toro)")
ax.set_ylabel("Ψ₀")
ax.set_title("E7 — Ψ₀ → 0 cuando L → ∞")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")

# E8 — cos(θ) como espectro
ax = axes[0, 2]
theta = np.linspace(0, 2*np.pi, 500)
ax.plot(theta, np.cos(theta), color="#9B59B6", lw=2)
ax.axhline(0, color="black", ls="--", lw=1)
ax.axhline(-1, color="#E74C3C", ls=":", lw=1, alpha=0.5)
ax.axhline(1, color="#E74C3C", ls=":", lw=1, alpha=0.5)
ax.fill_between(theta, -1, 1, alpha=0.1, color="#E74C3C")
ax.set_xlabel("θ")
ax.set_ylabel("cos(θ)")
ax.set_title("E8 — cos(θ) ∈ [−1, 1]")
ax.grid(alpha=0.3)

# E9 — identidad
ax = axes[1, 0]
ax.axis("off")
ax.text(0.5, 0.5,
        "E9:  ⟨0|0⟩/cos(0) = 1/1 = 1\n\n"
        "Identidad trivial\n"
        "Consecuencia de E2 ∧ E8\n\n"
        "No aporta contenido nuevo.",
        ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round", facecolor="#F4F4F4",
                  edgecolor="#333", linewidth=1.5))

# E10 — idempotencia
ax = axes[1, 1]
casos = ["‖v‖=1\n(E2 ✓)", "‖v‖=2\n(E2 ✗)", "‖v‖=0.5\n(E2 ✗)"]
valores = [f1, f2, f05]
colors = ["#27AE60" if v > 0.5 else "#E74C3C" for v in valores]
bars = ax.bar(casos, valores, color=colors, edgecolor="black")
for bar, v in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.3f}",
            ha="center", fontsize=10)
ax.set_ylabel("Fracción con ℛ² = ℛ")
ax.set_title("E10 — Idempotencia depende de E2")
ax.set_ylim(0, 1.1)
ax.grid(alpha=0.3, axis="y")

# Resumen
ax = axes[1, 2]
ax.axis("off")
resumen = """
BATCH E6-E10 · RESUMEN

E6  [PENDIENTE]        falta fórmula
E7  [POSTULADO]        compatible E2, C-006 activo
E8  [PENDIENTE-AMBIGUO] ¿operador o proyector?
E9  [IDENTIDAD-TRIVIAL] E2 ∧ E8 → E9
E10 [ESTRUCTURAL]      ℛ² = ℛ ⟺ E2

Dependencias:
E9 → E2, E8
E10 → E2
"""
ax.text(0.02, 0.98, resumen, transform=ax.transAxes,
        fontsize=9.5, va="top", family="monospace",
        bbox=dict(boxstyle="round", facecolor="#F4F4F4",
                  edgecolor="#333", linewidth=1.5))

plt.suptitle("RG-108 · Batch E6-E10 · AA + Monte Carlo", fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig("batch_E6_E10.png", dpi=140, bbox_inches="tight")
plt.show()

print("Guardado: batch_E6_E10.png")
print()

# ============================================================
# RESUMEN FINAL
# ============================================================
print("=" * 76)
print("RESUMEN BATCH E6-E10")
print("=" * 76)
print("""
E6 · V(φ) = (𝔇/N)[...]
  → [PENDIENTE] · falta fórmula completa.
  → Compatible con E4 si V(0)=0 y V≥0.

E7 · Ψ₀ = 1/√Vol(T^N)
  → [POSTULADO] · normalización OK para Vol finito.
  → C-006 activo si Vol → ∞.

E8 · P̂(θ) = cos(θ)
  → [PENDIENTE-AMBIGUO] · ¿operador multiplicativo o proyector?
  → Si multiplicador: P̂(0) = 𝕀 → E3 se cumple trivialmente.

E9 · ⟨0|0⟩/cos(0) = 1
  → [IDENTIDAD-TRIVIAL] · consecuencia de E2 ∧ E8.
  → No aporta contenido nuevo.

E10 · ℛ = |0⟩⟨0|
  → [ESTRUCTURAL] · ℛ² = ℛ ⟺ E2.
  → E2 no es decorativa, HABILITA E10.

DEPENDENCIAS NUEVAS:
  E9 → E2, E8
  E10 → E2
  E6 → E4 (si V(0)=0)
  E3 → E8 (si P̂(0)=𝕀)
""")

In [ ]:
# =============================================================================
# SIMULACIÓN GLOBAL RG — INSTALACIÓN Y CONFIGURACIÓN
# Ejecutable en Google Colab
# =============================================================================

# @title 📦 Instalación de dependencias
!pip install numpy scipy matplotlib pandas tqdm > /dev/null 2>&1

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy import stats
from scipy.fft import fft, fftfreq
import pandas as pd
from tqdm import tqdm
import json
import time
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Callable

# Configuración estética
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Dependencias instaladas")
print(f"📅 Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
import numpy as np
from scipy.spatial import KDTree

# =============================================================================
# CAPA 0: INFRAESTRUCTURA BASE
# =============================================================================

# --- Constantes fundamentales del corpus RG ---
class RGConstants:
    """Constantes fundamentales derivadas del corpus."""
    D_DEUDA = 1.5              # Deuda de Información
    ETA = np.pi / 57           # Fricción topológica
    LAMBDA = 1.0 / 18          # Acoplamiento fuerte
    ALPHA_INV = 137.03630378   # α⁻¹ = 4π³ + π² + π
    N19 = 19                   # Clúster electrón
    N57 = 57                   # Hard-Lock protón
    A_CANON = 1.0
    B_CANON = 1.0
    C_CANON = 2.0
    a_CANON = 0.5
    b_CANON = 0.5
    c_CANON = 1.0

# --- Red Hexagonal Genérica ---
class RedHexagonal:
    """Red hexagonal con parámetros ajustables."""

    def __init__(self, N=19, J=1.0, D=1.5, lambda_int=0.0):
        self.N = N
        self.J = J
        self.D = D
        self.lambda_int = lambda_int
        self.fases = np.zeros(N)
        self.posiciones = self._generar_posiciones()
        self.vecinos = self._encontrar_vecinos()
        self.matriz_adyacencia = self._construir_adyacencia()

    def _generar_posiciones(self):
        if self.N == 19:
            pos = [(0.0, 0.0)]
            for i in range(6):
                ang = i * np.pi / 3
                pos.append((np.cos(ang), np.sin(ang)))
            for i in range(12):
                ang = i * np.pi / 6 + np.pi / 12
                pos.append((2*np.cos(ang), 2*np.sin(ang)))
            return np.array(pos)
        elif self.N == 57:
            pos = []
            for k in range(3):
                cx = 3 * np.cos(2*k*np.pi/3 + np.pi/6)
                cy = 3 * np.sin(2*k*np.pi/3 + np.pi/6)
                for i in range(19):
                    if i == 0:
                        pos.append((cx, cy))
                    elif i < 7:
                        ang = (i-1) * np.pi/3
                        pos.append((cx + np.cos(ang), cy + np.sin(ang)))
                    else:
                        ang = (i-7) * np.pi/6 + np.pi/12
                        pos.append((cx + 2*np.cos(ang), cy + 2*np.sin(ang)))
            return np.array(pos)
        else:
            L = int(np.ceil(np.sqrt(self.N)))
            pos = []
            for i in range(L):
                for j in range(L):
                    x = i * 1.5
                    y = j * np.sqrt(3) + (i % 2) * np.sqrt(3) / 2
                    pos.append((x, y))
            return np.array(pos[:self.N])

    def _encontrar_vecinos(self, radio=1.5):
        kdtree = KDTree(self.posiciones)
        vecinos = []
        for i in range(self.N):
            idx = kdtree.query_ball_point(self.posiciones[i], r=radio)
            vecinos.append([j for j in idx if j != i])
        return vecinos

    def _construir_adyacencia(self):
        adj = np.zeros((self.N, self.N))
        for i in range(self.N):
            for j in self.vecinos[i]:
                adj[i, j] = 1.0
        return adj

    def energia_XY(self):
        E = 0.0
        for i in range(self.N):
            for j in self.vecinos[i]:
                if i < j:
                    delta = self.fases[i] - self.fases[j]
                    E += 0.5 * self.J * (1 - np.cos(delta))
        return E

    def energia_RG(self, usar_potencial=True):
        E = self.energia_XY()
        if usar_potencial:
            phi = np.mean(self.fases)
            x = phi / np.pi
            V = self.D * (x**4 - 2*x**2 + 1)
            E += V
        return E

    def medir_roles(self):
        """Mide los Seis Roles a partir de las fases."""
        fases = self.fases
        N = len(fases)

        # FFT de las fases (para detectar modos)
        fft_fases = np.fft.fft(np.exp(1j * fases))
        amplitudes = np.abs(fft_fases) / N

        # Roles via FFT (versión alternativa)
        A_fft = amplitudes[0]
        B_fft = amplitudes[1] if N > 1 else 0.0

        # Roles via media de fases
        theta_prom = np.mean(fases)
        A = 1.0 + np.sin(theta_prom)
        B = 1.0 - np.sin(theta_prom)
        T = A + B
        a = A / T if T > 1e-10 else 0.5
        b = B / T if T > 1e-10 else 0.5
        c = 1.0
        C = A / a if a > 1e-10 else 2.0

        return {
            'A': A, 'B': B, 'C': C, 'a': a, 'b': b, 'c': c, 'T': T,
            'A_fft': A_fft, 'B_fft': B_fft
        }

    def medir_deuda(self):
        r = self.medir_roles()
        return r['A'] * r['B'] * r['C'] - 2 * r['a'] * r['b'] * r['c']

    def paso_monte_carlo(self, delta=0.1, T=0.01, usar_RG=True):
        i = np.random.randint(0, self.N)
        theta_old = self.fases[i]
        theta_new = theta_old + np.random.uniform(-delta, delta)

        E_old = sum(self.J * (1 - np.cos(theta_old - self.fases[j]))
                    for j in self.vecinos[i])
        E_new = sum(self.J * (1 - np.cos(theta_new - self.fases[j]))
                    for j in self.vecinos[i])
        dE = E_new - E_old

        if usar_RG:
            phi_old = np.mean(self.fases)
            phi_new = phi_old + (theta_new - theta_old) / self.N
            x_old, x_new = phi_old/np.pi, phi_new/np.pi
            dV = self.D * ((x_new**4 - 2*x_new**2) - (x_old**4 - 2*x_old**2))
            dE += dV

        if dE < 0 or np.random.random() < np.exp(-dE / max(T, 1e-10)):
            self.fases[i] = theta_new
            return True
        return False

print("✅ Red hexagonal definida")

In [ ]:
# =============================================================================
# MOTOR MONTECARLO GLOBAL
# =============================================================================

@dataclass
class ResultadoSimulacion:
    """Resultado estructurado de una simulación."""
        fase: int
            nombre: str
                N: int
                    T: float
                        pasos: int
                            energia_media: float
                                energia_std: float
                                    deuda_media: float
                                        deuda_std: float
                                            roles_media: Dict[str, float]
                                                tasa_aceptacion: float
                                                    tiempo_simulacion: float
                                                        anomalias_detectadas: int

                                                                def to_dict(self):
                                                                        return asdict(self)

                                                                        def simular_fase(
                                                                            fase: int,
                                                                                nombre: str,
                                                                                    N: int = 19,
                                                                                        T: float = 0.01,
                                                                                            pasos: int = 100000,
                                                                                                delta: float = 0.1,
                                                                                                    usar_RG: bool = True,
                                                                                                        usar_ruptura: bool = False,
                                                                                                            seed: int = None
                                                                                                            ) -> ResultadoSimulacion:
                                                                                                                """
                                                                                                                    Ejecuta una simulación Monte Carlo completa de una fase.
                                                                                                                        """
                                                                                                                            if seed is not None:
                                                                                                                                    np.random.seed(seed)

                                                                                                                                            inicio = time.time()

                                                                                                                                                    red = RedHexagonal(N=N, D=RGConstants.D_DEUDA if usar_RG else 0.0)
                                                                                                                                                        red.fases = np.zeros(N)

                                                                                                                                                                # Perturbación inicial opcional
                                                                                                                                                                    if usar_ruptura:
                                                                                                                                                                            red.fases[0] = 0.1
                                                                                                                                                                                    if N > 1:
                                                                                                                                                                                                red.fases[1] = -0.1

                                                                                                                                                                                                        hist_energia = []
                                                                                                                                                                                                            hist_deuda = []
                                                                                                                                                                                                                hist_roles = {'A': [], 'B': [], 'C': [], 'a': [], 'b': [], 'c': []}
                                                                                                                                                                                                                    aceptados = 0

                                                                                                                                                                                                                            intervalo = max(1, pasos // 1000)

                                                                                                                                                                                                                                    for paso in range(pasos):
                                                                                                                                                                                                                                            if red.paso_monte_carlo(delta=delta, T=T, usar_RG=usar_RG):
                                                                                                                                                                                                                                                        aceptados += 1

                                                                                                                                                                                                                                                                        if paso % intervalo == 0:
                                                                                                                                                                                                                                                                                    hist_energia.append(red.energia_RG(usar_potencial=usar_RG))
                                                                                                                                                                                                                                                                                                hist_deuda.append(red.medir_deuda())
                                                                                                                                                                                                                                                                                                            roles = red.medir_roles()
                                                                                                                                                                                                                                                                                                                        for k in hist_roles:
                                                                                                                                                                                                                                                                                                                                        hist_roles[k].append(roles[k])

                                                                                                                                                                                                                                                                                                                                                tiempo = time.time() - inicio

                                                                                                                                                                                                                                                                                                                                                        # Análisis de resultados
                                                                                                                                                                                                                                                                                                                                                            n_eq = min(500, len(hist_energia))
                                                                                                                                                                                                                                                                                                                                                                E_media = np.mean(hist_energia[-n_eq:])
                                                                                                                                                                                                                                                                                                                                                                    E_std = np.std(hist_energia[-n_eq:])
                                                                                                                                                                                                                                                                                                                                                                        D_media = np.mean(hist_deuda[-n_eq:])
                                                                                                                                                                                                                                                                                                                                                                            D_std = np.std(hist_deuda[-n_eq:])

                                                                                                                                                                                                                                                                                                                                                                                    roles_media = {k: np.mean(v[-n_eq:]) for k, v in hist_roles.items()}

                                                                                                                                                                                                                                                                                                                                                                                            # Anomaly Analysis
                                                                                                                                                                                                                                                                                                                                                                                                anomalias = detectar_anomalias(hist_deuda, hist_energia)

                                                                                                                                                                                                                                                                                                                                                                                                        return ResultadoSimulacion(
                                                                                                                                                                                                                                                                                                                                                                                                                fase=fase,
                                                                                                                                                                                                                                                                                                                                                                                                                        nombre=nombre,
                                                                                                                                                                                                                                                                                                                                                                                                                                N=N,
                                                                                                                                                                                                                                                                                                                                                                                                                                        T=T,
                                                                                                                                                                                                                                                                                                                                                                                                                                                pasos=pasos,
                                                                                                                                                                                                                                                                                                                                                                                                                                                        energia_media=E_media,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                energia_std=E_std,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                        deuda_media=D_media,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                deuda_std=D_std,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        roles_media=roles_media,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                tasa_aceptacion=aceptados / pasos,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        tiempo_simulacion=tiempo,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                anomalias_detectadas=anomalias
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    )

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    print("✅ Motor Monte Carlo definido")